In [0]:
telecom_table = "workspace.census360.silver_cbsl_telecommunication_services"

if spark.catalog.tableExists(telecom_table):
    cbsl_telecom_silver_df = spark.table(telecom_table)

    print(f"Table: {telecom_table}")
    print("Exists: True")
    print(f"Row count: {cbsl_telecom_silver_df.count()}")
    print(f"Column count: {len(cbsl_telecom_silver_df.columns)}")
    print("\nColumns:")
    print(cbsl_telecom_silver_df.columns)

    print("\nSchema:")
    cbsl_telecom_silver_df.printSchema()
else:
    print(f"Table does not exist: {telecom_table}")

In [0]:
from pyspark.sql import functions as F

print("Year coverage:")
cbsl_telecom_silver_df.select(
    F.min("year").alias("min_year"),
    F.max("year").alias("max_year"),
    F.countDistinct("year").alias("distinct_years")
).show()

print("Rows per year:")
cbsl_telecom_silver_df.groupBy("year") \
    .count() \
    .orderBy("year") \
    .show(20, truncate=False)

print("Metrics and observation counts:")
cbsl_telecom_silver_df.groupBy("metric_code", "metric_label") \
    .agg(
        F.count("*").alias("row_count"),
        F.countDistinct("year").alias("year_count"),
        F.sum(F.col("value_numeric").isNull().cast("int")).alias("numeric_null_count")
    ) \
    .orderBy("metric_code") \
    .show(50, truncate=False)

print("Value status counts:")
cbsl_telecom_silver_df.groupBy("value_status") \
    .count() \
    .orderBy("value_status") \
    .show(truncate=False)

print("Data status counts:")
cbsl_telecom_silver_df.groupBy("data_status") \
    .count() \
    .orderBy("data_status") \
    .show(truncate=False)

print("Duplicate metric-year keys:")
duplicate_metric_year_df = (
    cbsl_telecom_silver_df
    .groupBy("metric_code", "year")
    .count()
    .filter(F.col("count") > 1)
)

print(f"Duplicate key count: {duplicate_metric_year_df.count()}")
duplicate_metric_year_df.orderBy("metric_code", "year").show(truncate=False)

In [0]:
hies_table = "workspace.census360.silver_cbsl_household_income_hies"

if spark.catalog.tableExists(hies_table):
    cbsl_hies_silver_df = spark.table(hies_table)

    print(f"Table: {hies_table}")
    print("Exists: True")
    print(f"Row count: {cbsl_hies_silver_df.count()}")
    print(f"Column count: {len(cbsl_hies_silver_df.columns)}")

    print("\nColumns:")
    print(cbsl_hies_silver_df.columns)

    print("\nSchema:")
    cbsl_hies_silver_df.printSchema()
else:
    print(f"Table does not exist: {hies_table}")

In [0]:
from pyspark.sql import functions as F

print("Survey-period coverage:")
cbsl_hies_silver_df.select(
    F.min("period_start_year").alias("min_start_year"),
    F.max("period_end_year").alias("max_end_year"),
    F.countDistinct("survey_period").alias("distinct_periods")
).show()

print("Survey periods:")
cbsl_hies_silver_df.select(
    "survey_period",
    "survey_period_type",
    "period_start_year",
    "period_end_year"
).distinct().orderBy("period_start_year", "period_end_year").show(20, truncate=False)

print("Metrics and observation counts:")
cbsl_hies_silver_df.groupBy("metric_code", "metric_label", "unit") \
    .agg(
        F.count("*").alias("row_count"),
        F.countDistinct("survey_period").alias("period_count"),
        F.sum(F.col("value_numeric").isNull().cast("int")).alias("numeric_null_count")
    ) \
    .orderBy("metric_code") \
    .show(50, truncate=False)

print("Value status counts:")
cbsl_hies_silver_df.groupBy("value_status") \
    .count() \
    .orderBy("value_status") \
    .show(truncate=False)

print("Duplicate metric-period keys:")
duplicate_hies_keys_df = (
    cbsl_hies_silver_df
    .groupBy("metric_code", "survey_period")
    .count()
    .filter(F.col("count") > 1)
)

print(f"Duplicate key count: {duplicate_hies_keys_df.count()}")
duplicate_hies_keys_df.orderBy("metric_code", "survey_period").show(truncate=False)

In [0]:
poverty_table = "workspace.census360.silver_cbsl_poverty_indicators"

if spark.catalog.tableExists(poverty_table):
    cbsl_poverty_silver_df = spark.table(poverty_table)

    print(f"Table: {poverty_table}")
    print("Exists: True")
    print(f"Row count: {cbsl_poverty_silver_df.count()}")
    print(f"Column count: {len(cbsl_poverty_silver_df.columns)}")

    print("\nColumns:")
    print(cbsl_poverty_silver_df.columns)

    print("\nSchema:")
    cbsl_poverty_silver_df.printSchema()
else:
    print(f"Table does not exist: {poverty_table}")

In [0]:
from pyspark.sql import functions as F

print("Geography-type summary:")
cbsl_poverty_silver_df.groupBy("geography_type") \
    .agg(
        F.countDistinct("geography_name").alias("geography_count"),
        F.count("*").alias("row_count")
    ) \
    .orderBy("geography_type") \
    .show(truncate=False)

print("Survey periods and definition variants:")
cbsl_poverty_silver_df.select(
    "survey_period",
    "survey_period_type",
    "period_start_year",
    "period_end_year",
    "definition_variant"
).distinct().orderBy(
    "period_end_year",
    "definition_variant"
).show(20, truncate=False)

print("Metrics and observation counts:")
cbsl_poverty_silver_df.groupBy("metric_code", "metric_label") \
    .agg(
        F.count("*").alias("row_count"),
        F.countDistinct("geography_name").alias("geography_count"),
        F.sum(F.col("value_numeric").isNull().cast("int")).alias("numeric_null_count")
    ) \
    .orderBy("metric_code") \
    .show(20, truncate=False)

print("Value status counts:")
cbsl_poverty_silver_df.groupBy("value_status") \
    .count() \
    .orderBy("value_status") \
    .show(truncate=False)

print("Unavailable poverty records:")
cbsl_poverty_silver_df.filter(
    F.col("value_numeric").isNull()
).select(
    "geography_type",
    "geography_name",
    "survey_period",
    "definition_variant",
    "metric_code",
    "value_status"
).orderBy(
    "geography_type",
    "geography_name",
    "survey_period",
    "metric_code"
).show(20, truncate=False)

print("Duplicate analytical keys:")
duplicate_poverty_keys_df = (
    cbsl_poverty_silver_df
    .groupBy(
        "geography_type",
        "geography_name",
        "survey_period",
        "definition_variant",
        "metric_code"
    )
    .count()
    .filter(F.col("count") > 1)
)

print(f"Duplicate key count: {duplicate_poverty_keys_df.count()}")
duplicate_poverty_keys_df.show(truncate=False)

In [0]:
provincial_table = "workspace.census360.silver_cbsl_provincial_socioeconomic"

if spark.catalog.tableExists(provincial_table):
    cbsl_provincial_silver_df = spark.table(provincial_table)

    print(f"Table: {provincial_table}")
    print("Exists: True")
    print(f"Row count: {cbsl_provincial_silver_df.count()}")
    print(f"Column count: {len(cbsl_provincial_silver_df.columns)}")

    print("\nColumns:")
    print(cbsl_provincial_silver_df.columns)

    print("\nSchema:")
    cbsl_provincial_silver_df.printSchema()
else:
    print(f"Table does not exist: {provincial_table}")

In [0]:
from pyspark.sql import functions as F

print("Year coverage:")
cbsl_provincial_silver_df.select(
    F.min("analysis_year").alias("min_year"),
    F.max("analysis_year").alias("max_year"),
    F.countDistinct("analysis_year").alias("distinct_years")
).show()

print("Geography-type summary:")
cbsl_provincial_silver_df.groupBy("geography_type") \
    .agg(
        F.countDistinct("geography_name").alias("geography_count"),
        F.count("*").alias("row_count")
    ) \
    .orderBy("geography_type") \
    .show(truncate=False)

print("Geographies:")
cbsl_provincial_silver_df.select(
    "geography_type",
    "geography_name"
).distinct().orderBy(
    "geography_type",
    "geography_name"
).show(20, truncate=False)

print("Section summary:")
cbsl_provincial_silver_df.groupBy("section_name") \
    .agg(
        F.countDistinct("metric_code").alias("metric_count"),
        F.count("*").alias("row_count")
    ) \
    .orderBy("section_name") \
    .show(truncate=False)

print("Overall metric and value summary:")
cbsl_provincial_silver_df.agg(
    F.countDistinct("metric_code").alias("distinct_metrics"),
    F.sum(F.col("value_numeric").isNull().cast("int")).alias("numeric_null_count"),
    F.countDistinct("value_status").alias("distinct_value_statuses")
).show()

print("Value status counts:")
cbsl_provincial_silver_df.groupBy("value_status") \
    .count() \
    .orderBy("value_status") \
    .show(truncate=False)

print("Duplicate analytical keys:")
duplicate_provincial_keys_df = (
    cbsl_provincial_silver_df
    .groupBy(
        "analysis_year",
        "geography_type",
        "geography_name",
        "metric_code"
    )
    .count()
    .filter(F.col("count") > 1)
)

print(f"Duplicate key count: {duplicate_provincial_keys_df.count()}")
duplicate_provincial_keys_df.show(20, truncate=False)

In [0]:
from pyspark.sql import functions as F

telecom_values_wide_df = (
    cbsl_telecom_silver_df
    .groupBy("year")
    .pivot("metric_code")
    .agg(F.first("value_numeric"))
)

telecom_year_status_df = (
    cbsl_telecom_silver_df
    .groupBy("year")
    .agg(
        F.first("data_status").alias("data_status"),
        F.countDistinct("data_status").alias("status_count")
    )
)

cbsl_telecom_annual_df = (
    telecom_values_wide_df
    .join(telecom_year_status_df, on="year", how="left")
    .orderBy("year")
)

print(f"Annual row count: {cbsl_telecom_annual_df.count()}")
print(f"Annual column count: {len(cbsl_telecom_annual_df.columns)}")

print("\nColumns:")
print(cbsl_telecom_annual_df.columns)

print("\nAnnual telecom data:")
cbsl_telecom_annual_df.show(20, truncate=False)

print("\nYear-status validation:")
cbsl_telecom_annual_df.select(
    "year",
    "data_status",
    "status_count"
).orderBy("year").show(20, truncate=False)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

telecom_year_window = Window.orderBy("year")

telecom_validation_df = (
    cbsl_telecom_annual_df
    .withColumn(
        "previous_year",
        F.lag("year").over(telecom_year_window)
    )
    .withColumn(
        "year_gap",
        F.when(
            F.col("previous_year").isNotNull(),
            F.col("year") - F.col("previous_year")
        )
    )
    .withColumn(
        "calculated_fixed_base_thousand",
        F.col("wireline_telephones_thousand")
        + F.col("wireless_local_loop_thousand")
    )
    .withColumn(
        "fixed_base_difference_thousand",
        F.round(
            F.col("fixed_subscriber_base_thousand")
            - F.col("calculated_fixed_base_thousand"),
            4
        )
    )
)

print("Year continuity and fixed-base consistency:")
telecom_validation_df.select(
    "year",
    "previous_year",
    "year_gap",
    "fixed_subscriber_base_thousand",
    "wireline_telephones_thousand",
    "wireless_local_loop_thousand",
    "calculated_fixed_base_thousand",
    "fixed_base_difference_thousand"
).orderBy("year").show(20, truncate=False)

print("Validation summary:")
telecom_validation_df.agg(
    F.sum(
        F.when(
            F.col("previous_year").isNotNull()
            & (F.col("year_gap") != 1),
            1
        ).otherwise(0)
    ).alias("non_continuous_year_count"),

    F.sum(
        F.when(
            F.abs(F.col("fixed_base_difference_thousand")) > 0.01,
            1
        ).otherwise(0)
    ).alias("fixed_base_mismatch_count"),

    F.sum(
        F.when(
            F.col("status_count") != 1,
            1
        ).otherwise(0)
    ).alias("inconsistent_status_year_count")
).show()

In [0]:
from pyspark.sql import functions as F

cbsl_telecom_silver_df = spark.table(
    "workspace.census360.silver_cbsl_telecommunication_services"
)

telecom_values_wide_df = (
    cbsl_telecom_silver_df
    .groupBy("year")
    .pivot("metric_code")
    .agg(F.first("value_numeric"))
)

telecom_year_status_df = (
    cbsl_telecom_silver_df
    .groupBy("year")
    .agg(
        F.first("data_status").alias("data_status"),
        F.countDistinct("data_status").alias("status_count")
    )
)

cbsl_telecom_annual_df = (
    telecom_values_wide_df
    .join(telecom_year_status_df, on="year", how="left")
    .orderBy("year")
)

print(f"Annual telecom DataFrame recreated: {cbsl_telecom_annual_df.count()} rows")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

telecom_year_window = Window.orderBy("year")

telecom_validation_df = (
    cbsl_telecom_annual_df
    .withColumn(
        "previous_year",
        F.lag("year").over(telecom_year_window)
    )
    .withColumn(
        "year_gap",
        F.when(
            F.col("previous_year").isNotNull(),
            F.col("year") - F.col("previous_year")
        )
    )
    .withColumn(
        "calculated_fixed_base_thousand",
        F.col("wireline_telephones_thousand")
        + F.col("wireless_local_loop_thousand")
    )
    .withColumn(
        "fixed_base_difference_thousand",
        F.round(
            F.col("fixed_subscriber_base_thousand")
            - F.col("calculated_fixed_base_thousand"),
            4
        )
    )
)

print("Year continuity and fixed-base consistency:")

telecom_validation_df.select(
    "year",
    "previous_year",
    "year_gap",
    "fixed_subscriber_base_thousand",
    "wireline_telephones_thousand",
    "wireless_local_loop_thousand",
    "calculated_fixed_base_thousand",
    "fixed_base_difference_thousand"
).orderBy("year").show(20, truncate=False)

print("Validation summary:")

telecom_validation_df.agg(
    F.sum(
        F.when(
            F.col("previous_year").isNotNull()
            & (F.col("year_gap") != 1),
            1
        ).otherwise(0)
    ).alias("non_continuous_year_count"),

    F.sum(
        F.when(
            F.abs(F.col("fixed_base_difference_thousand")) > 0.01,
            1
        ).otherwise(0)
    ).alias("fixed_base_mismatch_count"),

    F.sum(
        F.when(
            F.col("status_count") != 1,
            1
        ).otherwise(0)
    ).alias("inconsistent_status_year_count")
).show()

In [0]:
from pyspark.sql import functions as F

cbsl_telecom_silver_df.filter(
    (F.col("year").isin(2021, 2024))
    & (
        F.col("metric_code").isin(
            "fixed_subscriber_base_thousand",
            "wireline_telephones_thousand",
            "wireless_local_loop_thousand"
        )
    )
).select(
    "year",
    "metric_code",
    "metric_label",
    "value_raw",
    "value_numeric",
    "data_status",
    "source_row_number"
).orderBy(
    "year",
    "metric_code"
).show(20, truncate=False)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

telecom_year_window = Window.orderBy("year")

cbsl_telecom_features_df = (
    cbsl_telecom_annual_df

    # Previous-year levels
    .withColumn(
        "internet_connections_previous_year",
        F.lag("internet_connections_total", 1).over(telecom_year_window)
    )
    .withColumn(
        "fixed_subscriber_base_previous_year_thousand",
        F.lag("fixed_subscriber_base_thousand", 1).over(telecom_year_window)
    )

    # Absolute annual changes
    .withColumn(
        "internet_connections_yoy_change",
        F.col("internet_connections_total")
        - F.col("internet_connections_previous_year")
    )
    .withColumn(
        "fixed_subscriber_base_yoy_change_thousand",
        F.col("fixed_subscriber_base_thousand")
        - F.col("fixed_subscriber_base_previous_year_thousand")
    )

    # Percentage annual changes with safe division
    .withColumn(
        "internet_connections_yoy_growth_percent",
        F.when(
            F.col("internet_connections_previous_year").isNotNull()
            & (F.col("internet_connections_previous_year") != 0),
            F.round(
                (
                    F.col("internet_connections_yoy_change")
                    / F.col("internet_connections_previous_year")
                ) * 100,
                4
            )
        )
    )
    .withColumn(
        "fixed_subscriber_base_yoy_growth_calculated_percent",
        F.when(
            F.col("fixed_subscriber_base_previous_year_thousand").isNotNull()
            & (F.col("fixed_subscriber_base_previous_year_thousand") != 0),
            F.round(
                (
                    F.col("fixed_subscriber_base_yoy_change_thousand")
                    / F.col("fixed_subscriber_base_previous_year_thousand")
                ) * 100,
                4
            )
        )
    )

    # Component consistency flag
    .withColumn(
        "fixed_component_difference_thousand",
        F.col("fixed_subscriber_base_thousand")
        - (
            F.col("wireline_telephones_thousand")
            + F.col("wireless_local_loop_thousand")
        )
    )
    .withColumn(
        "fixed_component_match_flag",
        F.when(
            F.abs(F.col("fixed_component_difference_thousand")) <= 0.01,
            F.lit(True)
        ).otherwise(F.lit(False))
    )

    .orderBy("year")
)

cbsl_telecom_features_df.select(
    "year",
    "internet_connections_total",
    "internet_connections_previous_year",
    "internet_connections_yoy_change",
    "internet_connections_yoy_growth_percent",
    "fixed_subscriber_base_thousand",
    "fixed_subscriber_growth_percent",
    "fixed_subscriber_base_yoy_growth_calculated_percent",
    "fixed_component_difference_thousand",
    "fixed_component_match_flag",
    "data_status"
).show(20, truncate=False)

In [0]:
from pyspark.sql import functions as F

telecom_feature_validation_df = (
    cbsl_telecom_features_df
    .withColumn(
        "fixed_growth_difference_percentage_points",
        F.round(
            F.col("fixed_subscriber_growth_percent")
            - F.col("fixed_subscriber_base_yoy_growth_calculated_percent"),
            4
        )
    )
)

print("Published versus calculated fixed-subscriber growth:")

telecom_feature_validation_df.select(
    "year",
    "fixed_subscriber_growth_percent",
    "fixed_subscriber_base_yoy_growth_calculated_percent",
    "fixed_growth_difference_percentage_points",
    "internet_connections_yoy_growth_percent",
    "data_status"
).orderBy("year").show(20, truncate=False)

print("Engineered-feature validation summary:")

telecom_feature_validation_df.agg(
    F.sum(
        F.col("internet_connections_yoy_growth_percent").isNull().cast("int")
    ).alias("internet_growth_null_count"),

    F.sum(
        F.col(
            "fixed_subscriber_base_yoy_growth_calculated_percent"
        ).isNull().cast("int")
    ).alias("calculated_fixed_growth_null_count"),

    F.max(
        F.abs(F.col("fixed_growth_difference_percentage_points"))
    ).alias("maximum_fixed_growth_difference_pp"),

    F.min(
        "internet_connections_yoy_growth_percent"
    ).alias("minimum_internet_growth_percent"),

    F.max(
        "internet_connections_yoy_growth_percent"
    ).alias("maximum_internet_growth_percent")
).show(truncate=False)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

telecom_year_window = Window.orderBy("year")

cbsl_telecom_features_v2_df = (
    cbsl_telecom_features_df

    # Previous-year penetration values
    .withColumn(
        "telephone_penetration_previous_year",
        F.lag("telephone_penetration_per_100", 1).over(telecom_year_window)
    )
    .withColumn(
        "cellular_penetration_previous_year",
        F.lag("cellular_penetration_per_100", 1).over(telecom_year_window)
    )

    # Annual penetration changes in percentage points
    .withColumn(
        "telephone_penetration_change_pp",
        F.round(
            F.col("telephone_penetration_per_100")
            - F.col("telephone_penetration_previous_year"),
            4
        )
    )
    .withColumn(
        "cellular_penetration_change_pp",
        F.round(
            F.col("cellular_penetration_per_100")
            - F.col("cellular_penetration_previous_year"),
            4
        )
    )

    # Sum of fixed-access components
    .withColumn(
        "fixed_component_total_thousand",
        F.col("wireline_telephones_thousand")
        + F.col("wireless_local_loop_thousand")
    )

    # Component shares using the component total as denominator
    .withColumn(
        "wireline_share_of_fixed_components_percent",
        F.when(
            F.col("fixed_component_total_thousand") > 0,
            F.round(
                (
                    F.col("wireline_telephones_thousand")
                    / F.col("fixed_component_total_thousand")
                ) * 100,
                4
            )
        )
    )
    .withColumn(
        "wireless_local_loop_share_of_fixed_components_percent",
        F.when(
            F.col("fixed_component_total_thousand") > 0,
            F.round(
                (
                    F.col("wireless_local_loop_thousand")
                    / F.col("fixed_component_total_thousand")
                ) * 100,
                4
            )
        )
    )

    .orderBy("year")
)

cbsl_telecom_features_v2_df.select(
    "year",
    "telephone_penetration_per_100",
    "telephone_penetration_change_pp",
    "cellular_penetration_per_100",
    "cellular_penetration_change_pp",
    "wireline_telephones_thousand",
    "wireless_local_loop_thousand",
    "wireline_share_of_fixed_components_percent",
    "wireless_local_loop_share_of_fixed_components_percent",
    "data_status"
).show(20, truncate=False)

In [0]:
from pyspark.sql import functions as F

telecom_v2_validation_df = (
    cbsl_telecom_features_v2_df
    .withColumn(
        "fixed_component_share_total_percent",
        F.round(
            F.col("wireline_share_of_fixed_components_percent")
            + F.col("wireless_local_loop_share_of_fixed_components_percent"),
            4
        )
    )
)

print("Feature validation by year:")

telecom_v2_validation_df.select(
    "year",
    "telephone_penetration_change_pp",
    "cellular_penetration_change_pp",
    "wireline_share_of_fixed_components_percent",
    "wireless_local_loop_share_of_fixed_components_percent",
    "fixed_component_share_total_percent",
    "data_status"
).orderBy("year").show(20, truncate=False)

print("Validation summary:")

telecom_v2_validation_df.agg(
    F.sum(
        F.col("telephone_penetration_change_pp").isNull().cast("int")
    ).alias("telephone_change_null_count"),

    F.sum(
        F.col("cellular_penetration_change_pp").isNull().cast("int")
    ).alias("cellular_change_null_count"),

    F.sum(
        F.when(
            F.abs(F.col("fixed_component_share_total_percent") - 100) > 0.01,
            1
        ).otherwise(0)
    ).alias("invalid_share_total_count"),

    F.sum(
        F.when(
            (F.col("wireline_share_of_fixed_components_percent") < 0)
            | (F.col("wireline_share_of_fixed_components_percent") > 100)
            | (F.col("wireless_local_loop_share_of_fixed_components_percent") < 0)
            | (F.col("wireless_local_loop_share_of_fixed_components_percent") > 100),
            1
        ).otherwise(0)
    ).alias("out_of_range_share_count"),

    F.min(
        "fixed_component_share_total_percent"
    ).alias("minimum_share_total_percent"),

    F.max(
        "fixed_component_share_total_percent"
    ).alias("maximum_share_total_percent")
).show(truncate=False)

In [0]:
from pyspark.sql import functions as F

cbsl_telecom_features_v3_df = (
    cbsl_telecom_features_v2_df

    .withColumn(
        "log_internet_connections_total",
        F.when(
            F.col("internet_connections_total") > 0,
            F.round(F.log(F.col("internet_connections_total")), 6)
        )
    )

    .withColumn(
        "log_fixed_subscriber_base_thousand",
        F.when(
            F.col("fixed_subscriber_base_thousand") > 0,
            F.round(F.log(F.col("fixed_subscriber_base_thousand")), 6)
        )
    )

    .orderBy("year")
)

print("Telecom level and log-transformed features:")

cbsl_telecom_features_v3_df.select(
    "year",
    "internet_connections_total",
    "log_internet_connections_total",
    "fixed_subscriber_base_thousand",
    "log_fixed_subscriber_base_thousand",
    "data_status"
).show(20, truncate=False)

print("Log-feature validation:")

cbsl_telecom_features_v3_df.agg(
    F.sum(
        F.col("log_internet_connections_total").isNull().cast("int")
    ).alias("internet_log_null_count"),

    F.sum(
        F.col("log_fixed_subscriber_base_thousand").isNull().cast("int")
    ).alias("fixed_base_log_null_count"),

    F.min("log_internet_connections_total").alias("minimum_internet_log"),
    F.max("log_internet_connections_total").alias("maximum_internet_log"),

    F.min("log_fixed_subscriber_base_thousand").alias("minimum_fixed_base_log"),
    F.max("log_fixed_subscriber_base_thousand").alias("maximum_fixed_base_log")
).show(truncate=False)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

telecom_year_window = Window.orderBy("year")

cbsl_telecom_features_v4_df = (
    cbsl_telecom_features_v3_df

    # Total internet-connection lags
    .withColumn(
        "internet_connections_lag_1_year",
        F.lag("internet_connections_total", 1).over(telecom_year_window)
    )
    .withColumn(
        "internet_connections_lag_2_years",
        F.lag("internet_connections_total", 2).over(telecom_year_window)
    )

    # Internet-growth lags
    .withColumn(
        "internet_growth_percent_lag_1_year",
        F.lag("internet_connections_yoy_growth_percent", 1)
        .over(telecom_year_window)
    )
    .withColumn(
        "internet_growth_percent_lag_2_years",
        F.lag("internet_connections_yoy_growth_percent", 2)
        .over(telecom_year_window)
    )

    # Fixed-subscriber-base lags
    .withColumn(
        "fixed_subscriber_base_lag_1_year_thousand",
        F.lag("fixed_subscriber_base_thousand", 1)
        .over(telecom_year_window)
    )
    .withColumn(
        "fixed_subscriber_base_lag_2_years_thousand",
        F.lag("fixed_subscriber_base_thousand", 2)
        .over(telecom_year_window)
    )

    # Calculated fixed-growth lags
    .withColumn(
        "fixed_growth_percent_lag_1_year",
        F.lag(
            "fixed_subscriber_base_yoy_growth_calculated_percent",
            1
        ).over(telecom_year_window)
    )
    .withColumn(
        "fixed_growth_percent_lag_2_years",
        F.lag(
            "fixed_subscriber_base_yoy_growth_calculated_percent",
            2
        ).over(telecom_year_window)
    )

    .orderBy("year")
)

print("Telecom lag features:")

cbsl_telecom_features_v4_df.select(
    "year",
    "internet_connections_total",
    "internet_connections_lag_1_year",
    "internet_connections_lag_2_years",
    "internet_connections_yoy_growth_percent",
    "internet_growth_percent_lag_1_year",
    "internet_growth_percent_lag_2_years",
    "fixed_subscriber_base_thousand",
    "fixed_subscriber_base_lag_1_year_thousand",
    "fixed_subscriber_base_lag_2_years_thousand",
    "data_status"
).show(20, truncate=False)

print("Lag null-count validation:")

cbsl_telecom_features_v4_df.agg(
    F.sum(
        F.col("internet_connections_lag_1_year").isNull().cast("int")
    ).alias("internet_level_lag_1_null_count"),

    F.sum(
        F.col("internet_connections_lag_2_years").isNull().cast("int")
    ).alias("internet_level_lag_2_null_count"),

    F.sum(
        F.col("fixed_subscriber_base_lag_1_year_thousand")
        .isNull().cast("int")
    ).alias("fixed_level_lag_1_null_count"),

    F.sum(
        F.col("fixed_subscriber_base_lag_2_years_thousand")
        .isNull().cast("int")
    ).alias("fixed_level_lag_2_null_count")
).show(truncate=False)

In [0]:
from pyspark.sql import functions as F

print("Growth-lag availability by year:")

cbsl_telecom_features_v4_df.select(
    "year",
    "internet_connections_yoy_growth_percent",
    "internet_growth_percent_lag_1_year",
    "internet_growth_percent_lag_2_years",
    "fixed_subscriber_base_yoy_growth_calculated_percent",
    "fixed_growth_percent_lag_1_year",
    "fixed_growth_percent_lag_2_years",
    "data_status"
).orderBy("year").show(20, truncate=False)

print("Growth-lag null-count validation:")

cbsl_telecom_features_v4_df.agg(
    F.sum(
        F.col("internet_growth_percent_lag_1_year")
        .isNull().cast("int")
    ).alias("internet_growth_lag_1_null_count"),

    F.sum(
        F.col("internet_growth_percent_lag_2_years")
        .isNull().cast("int")
    ).alias("internet_growth_lag_2_null_count"),

    F.sum(
        F.col("fixed_growth_percent_lag_1_year")
        .isNull().cast("int")
    ).alias("fixed_growth_lag_1_null_count"),

    F.sum(
        F.col("fixed_growth_percent_lag_2_years")
        .isNull().cast("int")
    ).alias("fixed_growth_lag_2_null_count")
).show(truncate=False)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

three_year_window = (
    Window
    .orderBy("year")
    .rowsBetween(-2, 0)
)

cbsl_telecom_features_v5_df = (
    cbsl_telecom_features_v4_df

    .withColumn(
        "internet_growth_3yr_observation_count",
        F.count("internet_connections_yoy_growth_percent")
        .over(three_year_window)
    )
    .withColumn(
        "internet_growth_3yr_trailing_average_percent",
        F.when(
            F.col("internet_growth_3yr_observation_count") == 3,
            F.round(
                F.avg("internet_connections_yoy_growth_percent")
                .over(three_year_window),
                4
            )
        )
    )

    .withColumn(
        "fixed_growth_3yr_observation_count",
        F.count(
            "fixed_subscriber_base_yoy_growth_calculated_percent"
        ).over(three_year_window)
    )
    .withColumn(
        "fixed_growth_3yr_trailing_average_percent",
        F.when(
            F.col("fixed_growth_3yr_observation_count") == 3,
            F.round(
                F.avg(
                    "fixed_subscriber_base_yoy_growth_calculated_percent"
                ).over(three_year_window),
                4
            )
        )
    )

    .orderBy("year")
)

cbsl_telecom_features_v5_df.select(
    "year",
    "internet_connections_yoy_growth_percent",
    "internet_growth_3yr_observation_count",
    "internet_growth_3yr_trailing_average_percent",
    "fixed_subscriber_base_yoy_growth_calculated_percent",
    "fixed_growth_3yr_observation_count",
    "fixed_growth_3yr_trailing_average_percent",
    "data_status"
).show(20, truncate=False)

In [0]:
from pyspark.sql import functions as F

print("Three-year rolling-feature validation:")

cbsl_telecom_features_v5_df.agg(
    F.sum(
        F.col(
            "internet_growth_3yr_trailing_average_percent"
        ).isNull().cast("int")
    ).alias("internet_3yr_average_null_count"),

    F.sum(
        F.col(
            "fixed_growth_3yr_trailing_average_percent"
        ).isNull().cast("int")
    ).alias("fixed_3yr_average_null_count"),

    F.sum(
        F.when(
            (
                F.col("internet_growth_3yr_observation_count") < 3
            )
            & F.col(
                "internet_growth_3yr_trailing_average_percent"
            ).isNotNull(),
            1
        ).otherwise(0)
    ).alias("invalid_internet_partial_window_count"),

    F.sum(
        F.when(
            (
                F.col("fixed_growth_3yr_observation_count") < 3
            )
            & F.col(
                "fixed_growth_3yr_trailing_average_percent"
            ).isNotNull(),
            1
        ).otherwise(0)
    ).alias("invalid_fixed_partial_window_count"),

    F.min(
        "internet_growth_3yr_trailing_average_percent"
    ).alias("minimum_internet_3yr_average"),

    F.max(
        "internet_growth_3yr_trailing_average_percent"
    ).alias("maximum_internet_3yr_average"),

    F.min(
        "fixed_growth_3yr_trailing_average_percent"
    ).alias("minimum_fixed_3yr_average"),

    F.max(
        "fixed_growth_3yr_trailing_average_percent"
    ).alias("maximum_fixed_3yr_average")
).show(truncate=False)

In [0]:
from pyspark.sql import functions as F

cbsl_telecom_features_v6_df = (
    cbsl_telecom_features_v5_df

    # Publication-status flags
    .withColumn(
        "is_published_observation",
        F.col("data_status") == "published"
    )
    .withColumn(
        "is_revised_observation",
        F.col("data_status") == "revised"
    )
    .withColumn(
        "is_provisional_observation",
        F.col("data_status") == "provisional"
    )

    # Analysis-readiness flags
    .withColumn(
        "yoy_growth_features_available",
        F.col("internet_connections_yoy_growth_percent").isNotNull()
        & F.col(
            "fixed_subscriber_base_yoy_growth_calculated_percent"
        ).isNotNull()
    )
    .withColumn(
        "one_year_growth_lags_available",
        F.col("internet_growth_percent_lag_1_year").isNotNull()
        & F.col("fixed_growth_percent_lag_1_year").isNotNull()
    )
    .withColumn(
        "two_year_growth_lags_available",
        F.col("internet_growth_percent_lag_2_years").isNotNull()
        & F.col("fixed_growth_percent_lag_2_years").isNotNull()
    )
    .withColumn(
        "three_year_rolling_features_available",
        F.col(
            "internet_growth_3yr_trailing_average_percent"
        ).isNotNull()
        & F.col(
            "fixed_growth_3yr_trailing_average_percent"
        ).isNotNull()
    )

    # Interpretation metadata
    .withColumn(
        "internet_metric_scope",
        F.lit("total_internet_connections_including_mobile")
    )
    .withColumn(
        "fixed_metric_scope",
        F.lit("fixed_access_subscriber_base_not_fixed_broadband")
    )

    .orderBy("year")
)

print("Telecom analysis-readiness flags:")

cbsl_telecom_features_v6_df.select(
    "year",
    "data_status",
    "is_published_observation",
    "is_revised_observation",
    "is_provisional_observation",
    "yoy_growth_features_available",
    "one_year_growth_lags_available",
    "two_year_growth_lags_available",
    "three_year_rolling_features_available"
).show(20, truncate=False)

print("Readiness summary:")

cbsl_telecom_features_v6_df.agg(
    F.sum(
        F.col("is_published_observation").cast("int")
    ).alias("published_year_count"),

    F.sum(
        F.col("is_revised_observation").cast("int")
    ).alias("revised_year_count"),

    F.sum(
        F.col("is_provisional_observation").cast("int")
    ).alias("provisional_year_count"),

    F.sum(
        F.col("yoy_growth_features_available").cast("int")
    ).alias("yoy_ready_year_count"),

    F.sum(
        F.col("one_year_growth_lags_available").cast("int")
    ).alias("one_year_lag_ready_count"),

    F.sum(
        F.col("two_year_growth_lags_available").cast("int")
    ).alias("two_year_lag_ready_count"),

    F.sum(
        F.col("three_year_rolling_features_available").cast("int")
    ).alias("rolling_ready_year_count")
).show(truncate=False)

In [0]:
cbsl_telecom_analytical_df = (
    cbsl_telecom_features_v6_df
    .select(
        # Time and source status
        "year",
        "data_status",
        "is_published_observation",
        "is_revised_observation",
        "is_provisional_observation",

        # Original telecom levels
        "internet_connections_total",
        "fixed_subscriber_base_thousand",
        "wireline_telephones_thousand",
        "wireless_local_loop_thousand",
        "telephone_penetration_per_100",
        "cellular_phones_thousand",
        "cellular_penetration_per_100",
        "public_pay_phone_booths",

        # Annual changes and growth
        "internet_connections_yoy_change",
        "internet_connections_yoy_growth_percent",
        "fixed_subscriber_base_yoy_change_thousand",
        "fixed_subscriber_growth_percent",
        "fixed_subscriber_base_yoy_growth_calculated_percent",
        "telephone_penetration_change_pp",
        "cellular_penetration_change_pp",

        # Fixed-access composition
        "wireline_share_of_fixed_components_percent",
        "wireless_local_loop_share_of_fixed_components_percent",
        "fixed_component_difference_thousand",
        "fixed_component_match_flag",

        # Log-transformed levels
        "log_internet_connections_total",
        "log_fixed_subscriber_base_thousand",

        # One-year and two-year lags
        "internet_connections_lag_1_year",
        "internet_connections_lag_2_years",
        "internet_growth_percent_lag_1_year",
        "internet_growth_percent_lag_2_years",
        "fixed_subscriber_base_lag_1_year_thousand",
        "fixed_subscriber_base_lag_2_years_thousand",
        "fixed_growth_percent_lag_1_year",
        "fixed_growth_percent_lag_2_years",

        # Rolling descriptive features
        "internet_growth_3yr_trailing_average_percent",
        "fixed_growth_3yr_trailing_average_percent",

        # Analysis readiness
        "yoy_growth_features_available",
        "one_year_growth_lags_available",
        "two_year_growth_lags_available",
        "three_year_rolling_features_available",

        # Interpretation metadata
        "internet_metric_scope",
        "fixed_metric_scope"
    )
    .orderBy("year")
)

print(f"Curated telecom rows: {cbsl_telecom_analytical_df.count()}")
print(f"Curated telecom columns: {len(cbsl_telecom_analytical_df.columns)}")

print("\nColumns:")
print(cbsl_telecom_analytical_df.columns)

print("\nSchema:")
cbsl_telecom_analytical_df.printSchema()

In [0]:
from pyspark.sql import functions as F

print("Curated telecom final validation:")

duplicate_telecom_years_df = (
    cbsl_telecom_analytical_df
    .groupBy("year")
    .count()
    .filter(F.col("count") > 1)
)

cbsl_telecom_analytical_df.agg(
    F.count("*").alias("row_count"),
    F.countDistinct("year").alias("distinct_year_count"),
    F.min("year").alias("minimum_year"),
    F.max("year").alias("maximum_year"),

    F.sum(
        F.col("internet_connections_total").isNull().cast("int")
    ).alias("internet_level_null_count"),

    F.sum(
        F.col("fixed_subscriber_base_thousand").isNull().cast("int")
    ).alias("fixed_base_null_count"),

    F.sum(
        F.col("internet_metric_scope").isNull().cast("int")
    ).alias("internet_scope_null_count"),

    F.sum(
        F.col("fixed_metric_scope").isNull().cast("int")
    ).alias("fixed_scope_null_count")
).show(truncate=False)

print(f"Duplicate year count: {duplicate_telecom_years_df.count()}")
duplicate_telecom_years_df.orderBy("year").show(truncate=False)

print("Expected engineered-feature null counts:")

cbsl_telecom_analytical_df.agg(
    F.sum(
        F.col("internet_connections_yoy_growth_percent")
        .isNull().cast("int")
    ).alias("internet_yoy_null_count"),

    F.sum(
        F.col("internet_connections_lag_1_year")
        .isNull().cast("int")
    ).alias("internet_level_lag_1_null_count"),

    F.sum(
        F.col("internet_connections_lag_2_years")
        .isNull().cast("int")
    ).alias("internet_level_lag_2_null_count"),

    F.sum(
        F.col("internet_growth_3yr_trailing_average_percent")
        .isNull().cast("int")
    ).alias("internet_3yr_average_null_count")
).show(truncate=False)

In [0]:
from pyspark.sql import functions as F

cbsl_hies_silver_df = spark.table(
    "workspace.census360.silver_cbsl_household_income_hies"
)

hies_period_metadata_df = (
    cbsl_hies_silver_df
    .select(
        "survey_period",
        "survey_period_type",
        "period_start_year",
        "period_end_year"
    )
    .distinct()
)

hies_values_wide_df = (
    cbsl_hies_silver_df
    .groupBy("survey_period")
    .pivot("metric_code")
    .agg(F.first("value_numeric"))
)

cbsl_hies_survey_features_df = (
    hies_period_metadata_df
    .join(
        hies_values_wide_df,
        on="survey_period",
        how="inner"
    )
    .orderBy(
        "period_start_year",
        "period_end_year"
    )
)

print(f"HIES survey-feature rows: {cbsl_hies_survey_features_df.count()}")
print(f"HIES survey-feature columns: {len(cbsl_hies_survey_features_df.columns)}")

print("\nColumns:")
print(cbsl_hies_survey_features_df.columns)

print("\nSurvey-period feature data:")
cbsl_hies_survey_features_df.show(20, truncate=False)

In [0]:
from pyspark.sql import functions as F

duplicate_hies_periods_df = (
    cbsl_hies_survey_features_df
    .groupBy("survey_period")
    .count()
    .filter(F.col("count") > 1)
)

print("HIES survey-feature validation summary:")

cbsl_hies_survey_features_df.agg(
    F.count("*").alias("row_count"),
    F.countDistinct("survey_period").alias("distinct_period_count"),
    F.min("period_start_year").alias("minimum_start_year"),
    F.max("period_end_year").alias("maximum_end_year"),

    F.sum(
        F.col("mean_household_income_monthly_lkr").isNull().cast("int")
    ).alias("mean_income_null_count"),

    F.sum(
        F.col("median_household_income_monthly_lkr").isNull().cast("int")
    ).alias("median_income_null_count"),

    F.sum(
        F.col("mean_household_expenditure_monthly_lkr").isNull().cast("int")
    ).alias("household_expenditure_null_count"),

    F.sum(
        F.col("food_ratio_percent").isNull().cast("int")
    ).alias("food_ratio_null_count"),

    F.sum(
        F.col("gini_household_income").isNull().cast("int")
    ).alias("income_gini_null_count"),

    F.sum(
        F.col("poverty_headcount_ratio_2002_based_percent")
        .isNull().cast("int")
    ).alias("poverty_headcount_null_count")
).show(truncate=False)

print(f"Duplicate survey-period count: {duplicate_hies_periods_df.count()}")
duplicate_hies_periods_df.orderBy("survey_period").show(truncate=False)

print("Period metadata consistency:")

cbsl_hies_survey_features_df.select(
    "survey_period",
    "survey_period_type",
    "period_start_year",
    "period_end_year",
    (
        F.col("period_end_year") - F.col("period_start_year")
    ).alias("period_span_years")
).orderBy(
    "period_start_year",
    "period_end_year"
).show(20, truncate=False)

In [0]:
from pyspark.sql import functions as F

cbsl_hies_alignment_options_df = (
    cbsl_hies_survey_features_df

    # Exact survey-period end year
    .withColumn(
        "end_year_candidate",
        F.col("period_end_year")
    )

    # Midpoint year for comparison only
    .withColumn(
        "midpoint_year_candidate",
        F.round(
            (
                F.col("period_start_year")
                + F.col("period_end_year")
            ) / 2
        ).cast("int")
    )

    # Whether each candidate overlaps the telecom period 2013–2024
    .withColumn(
        "end_year_overlaps_telecom",
        F.col("end_year_candidate").between(2013, 2024)
    )
    .withColumn(
        "midpoint_year_overlaps_telecom",
        F.col("midpoint_year_candidate").between(2013, 2024)
    )

    # Explicitly identify original versus potentially derived alignment
    .withColumn(
        "end_year_alignment_type",
        F.lit("original_period_end_year")
    )
    .withColumn(
        "midpoint_alignment_type",
        F.when(
            F.col("survey_period_type") == "single_year",
            F.lit("original_single_year")
        ).otherwise(
            F.lit("derived_midpoint_year_for_comparison_only")
        )
    )

    .orderBy("period_start_year", "period_end_year")
)

print("HIES alignment-option comparison:")

cbsl_hies_alignment_options_df.select(
    "survey_period",
    "survey_period_type",
    "period_start_year",
    "period_end_year",
    "end_year_candidate",
    "end_year_overlaps_telecom",
    "midpoint_year_candidate",
    "midpoint_year_overlaps_telecom",
    "end_year_alignment_type",
    "midpoint_alignment_type"
).show(20, truncate=False)

print("Overlap summary:")

cbsl_hies_alignment_options_df.agg(
    F.sum(
        F.col("end_year_overlaps_telecom").cast("int")
    ).alias("end_year_overlap_period_count"),

    F.sum(
        F.col("midpoint_year_overlaps_telecom").cast("int")
    ).alias("midpoint_overlap_period_count"),

    F.sum(
        (
            F.col("survey_period_type") == "single_year"
        ).cast("int")
    ).alias("single_year_period_count"),

    F.sum(
        (
            F.col("survey_period_type") == "multi_year"
        ).cast("int")
    ).alias("multi_year_period_count")
).show(truncate=False)

In [0]:
from pyspark.sql import functions as F

cbsl_hies_aligned_features_df = (
    cbsl_hies_survey_features_df

    .withColumn(
        "representative_year",
        F.col("period_end_year")
    )

    .withColumn(
        "alignment_method",
        F.lit("survey_period_end_year")
    )

    .withColumn(
        "is_original_single_year_observation",
        F.col("survey_period_type") == "single_year"
    )

    .withColumn(
        "is_multi_year_survey_observation",
        F.col("survey_period_type") == "multi_year"
    )

    .withColumn(
        "is_derived_annual_value",
        F.lit(False)
    )

    .withColumn(
        "overlaps_cbsl_telecom_period",
        F.col("representative_year").between(2013, 2024)
    )

    .orderBy("representative_year")
)

print("Documented HIES alignment:")

cbsl_hies_aligned_features_df.select(
    "survey_period",
    "survey_period_type",
    "period_start_year",
    "period_end_year",
    "representative_year",
    "alignment_method",
    "is_original_single_year_observation",
    "is_multi_year_survey_observation",
    "is_derived_annual_value",
    "overlaps_cbsl_telecom_period"
).show(20, truncate=False)

print("Alignment validation summary:")

cbsl_hies_aligned_features_df.agg(
    F.count("*").alias("row_count"),

    F.countDistinct("representative_year")
    .alias("distinct_representative_year_count"),

    F.sum(
        F.col("overlaps_cbsl_telecom_period").cast("int")
    ).alias("telecom_overlap_observation_count"),

    F.sum(
        F.col("is_derived_annual_value").cast("int")
    ).alias("derived_annual_value_count")
).show(truncate=False)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

hies_timeline_window = Window.orderBy("representative_year")

cbsl_hies_timeline_df = (
    cbsl_hies_aligned_features_df

    .withColumn(
        "previous_survey_period",
        F.lag("survey_period", 1).over(hies_timeline_window)
    )

    .withColumn(
        "previous_representative_year",
        F.lag("representative_year", 1).over(hies_timeline_window)
    )

    .withColumn(
        "years_since_previous_survey",
        F.when(
            F.col("previous_representative_year").isNotNull(),
            F.col("representative_year")
            - F.col("previous_representative_year")
        )
    )

    .orderBy("representative_year")
)

print("HIES survey timeline and interval gaps:")

cbsl_hies_timeline_df.select(
    "survey_period",
    "survey_period_type",
    "representative_year",
    "previous_survey_period",
    "previous_representative_year",
    "years_since_previous_survey",
    "overlaps_cbsl_telecom_period"
).show(20, truncate=False)

print("Survey-gap validation summary:")

cbsl_hies_timeline_df.agg(
    F.sum(
        F.col("years_since_previous_survey").isNull().cast("int")
    ).alias("survey_gap_null_count"),

    F.min(
        "years_since_previous_survey"
    ).alias("minimum_survey_gap_years"),

    F.max(
        "years_since_previous_survey"
    ).alias("maximum_survey_gap_years"),

    F.sum(
        F.when(
            F.col("years_since_previous_survey") <= 0,
            1
        ).otherwise(0)
    ).alias("invalid_survey_gap_count")
).show(truncate=False)

In [0]:
cbsl_hies_selected_features_df = (
    cbsl_hies_timeline_df
    .select(
        # Survey metadata
        "survey_period",
        "survey_period_type",
        "period_start_year",
        "period_end_year",
        "representative_year",
        "previous_survey_period",
        "previous_representative_year",
        "years_since_previous_survey",
        "alignment_method",
        "is_original_single_year_observation",
        "is_multi_year_survey_observation",
        "is_derived_annual_value",
        "overlaps_cbsl_telecom_period",

        # Selected income indicators
        "mean_household_income_monthly_lkr",
        "median_household_income_monthly_lkr",
        "mean_per_capita_income_monthly_lkr",

        # Selected expenditure indicators
        "mean_household_expenditure_monthly_lkr",
        "food_drink_expenditure_monthly_lkr",
        "non_food_expenditure_monthly_lkr",
        "food_ratio_percent",

        # Distribution and poverty indicators
        "gini_household_income",
        "poverty_headcount_ratio_2002_based_percent",

        # Household structure
        "household_size",
        "income_receivers_per_household"
    )
    .orderBy("representative_year")
)

print(f"Selected HIES rows: {cbsl_hies_selected_features_df.count()}")
print(f"Selected HIES columns: {len(cbsl_hies_selected_features_df.columns)}")

print("\nSelected HIES candidate features:")

cbsl_hies_selected_features_df.select(
    "survey_period",
    "representative_year",
    "years_since_previous_survey",
    "mean_household_income_monthly_lkr",
    "median_household_income_monthly_lkr",
    "mean_per_capita_income_monthly_lkr",
    "mean_household_expenditure_monthly_lkr",
    "food_ratio_percent",
    "gini_household_income",
    "poverty_headcount_ratio_2002_based_percent",
    "overlaps_cbsl_telecom_period"
).show(20, truncate=False)

In [0]:
from pyspark.sql import functions as F

duplicate_selected_hies_years_df = (
    cbsl_hies_selected_features_df
    .groupBy("representative_year")
    .count()
    .filter(F.col("count") > 1)
)

print("Selected HIES feature validation:")

cbsl_hies_selected_features_df.agg(
    F.count("*").alias("row_count"),
    F.countDistinct("representative_year")
    .alias("distinct_representative_year_count"),

    F.sum(
        F.col("mean_household_income_monthly_lkr")
        .isNull().cast("int")
    ).alias("mean_income_null_count"),

    F.sum(
        F.col("median_household_income_monthly_lkr")
        .isNull().cast("int")
    ).alias("median_income_null_count"),

    F.sum(
        F.col("mean_per_capita_income_monthly_lkr")
        .isNull().cast("int")
    ).alias("per_capita_income_null_count"),

    F.sum(
        F.col("mean_household_expenditure_monthly_lkr")
        .isNull().cast("int")
    ).alias("household_expenditure_null_count"),

    F.sum(
        F.col("food_ratio_percent")
        .isNull().cast("int")
    ).alias("food_ratio_null_count"),

    F.sum(
        F.col("gini_household_income")
        .isNull().cast("int")
    ).alias("income_gini_null_count"),

    F.sum(
        F.col("poverty_headcount_ratio_2002_based_percent")
        .isNull().cast("int")
    ).alias("poverty_headcount_null_count"),

    F.sum(
        F.col("overlaps_cbsl_telecom_period").cast("int")
    ).alias("telecom_overlap_count")
).show(truncate=False)

print(f"Duplicate representative-year count: {duplicate_selected_hies_years_df.count()}")

duplicate_selected_hies_years_df.orderBy(
    "representative_year"
).show(truncate=False)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

cbsl_hies_silver_df = spark.table(
    "workspace.census360.silver_cbsl_household_income_hies"
)

hies_period_metadata_df = (
    cbsl_hies_silver_df
    .select(
        "survey_period",
        "survey_period_type",
        "period_start_year",
        "period_end_year"
    )
    .distinct()
)

hies_values_wide_df = (
    cbsl_hies_silver_df
    .groupBy("survey_period")
    .pivot("metric_code")
    .agg(F.first("value_numeric"))
)

hies_timeline_window = Window.orderBy("representative_year")

cbsl_hies_selected_features_df = (
    hies_period_metadata_df
    .join(
        hies_values_wide_df,
        on="survey_period",
        how="inner"
    )
    .withColumn(
        "representative_year",
        F.col("period_end_year")
    )
    .withColumn(
        "alignment_method",
        F.lit("survey_period_end_year")
    )
    .withColumn(
        "is_original_single_year_observation",
        F.col("survey_period_type") == "single_year"
    )
    .withColumn(
        "is_multi_year_survey_observation",
        F.col("survey_period_type") == "multi_year"
    )
    .withColumn(
        "is_derived_annual_value",
        F.lit(False)
    )
    .withColumn(
        "overlaps_cbsl_telecom_period",
        F.col("representative_year").between(2013, 2024)
    )
    .withColumn(
        "previous_survey_period",
        F.lag("survey_period", 1).over(hies_timeline_window)
    )
    .withColumn(
        "previous_representative_year",
        F.lag("representative_year", 1).over(hies_timeline_window)
    )
    .withColumn(
        "years_since_previous_survey",
        F.when(
            F.col("previous_representative_year").isNotNull(),
            F.col("representative_year")
            - F.col("previous_representative_year")
        )
    )
    .select(
        "survey_period",
        "survey_period_type",
        "period_start_year",
        "period_end_year",
        "representative_year",
        "previous_survey_period",
        "previous_representative_year",
        "years_since_previous_survey",
        "alignment_method",
        "is_original_single_year_observation",
        "is_multi_year_survey_observation",
        "is_derived_annual_value",
        "overlaps_cbsl_telecom_period",
        "mean_household_income_monthly_lkr",
        "median_household_income_monthly_lkr",
        "mean_per_capita_income_monthly_lkr",
        "mean_household_expenditure_monthly_lkr",
        "food_drink_expenditure_monthly_lkr",
        "non_food_expenditure_monthly_lkr",
        "food_ratio_percent",
        "gini_household_income",
        "poverty_headcount_ratio_2002_based_percent",
        "household_size",
        "income_receivers_per_household"
    )
    .orderBy("representative_year")
)

print(
    f"Selected HIES DataFrame recreated: "
    f"{cbsl_hies_selected_features_df.count()} rows, "
    f"{len(cbsl_hies_selected_features_df.columns)} columns"
)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

hies_change_window = Window.orderBy("representative_year")

cbsl_hies_change_features_df = (
    cbsl_hies_selected_features_df

    # Previous survey values
    .withColumn(
        "previous_mean_household_income_monthly_lkr",
        F.lag("mean_household_income_monthly_lkr", 1)
        .over(hies_change_window)
    )
    .withColumn(
        "previous_median_household_income_monthly_lkr",
        F.lag("median_household_income_monthly_lkr", 1)
        .over(hies_change_window)
    )
    .withColumn(
        "previous_mean_per_capita_income_monthly_lkr",
        F.lag("mean_per_capita_income_monthly_lkr", 1)
        .over(hies_change_window)
    )
    .withColumn(
        "previous_mean_household_expenditure_monthly_lkr",
        F.lag("mean_household_expenditure_monthly_lkr", 1)
        .over(hies_change_window)
    )
    .withColumn(
        "previous_food_ratio_percent",
        F.lag("food_ratio_percent", 1)
        .over(hies_change_window)
    )
    .withColumn(
        "previous_gini_household_income",
        F.lag("gini_household_income", 1)
        .over(hies_change_window)
    )
    .withColumn(
        "previous_poverty_headcount_percent",
        F.lag("poverty_headcount_ratio_2002_based_percent", 1)
        .over(hies_change_window)
    )

    # Total percentage change between surveys
    .withColumn(
        "mean_household_income_change_percent",
        F.when(
            F.col("previous_mean_household_income_monthly_lkr") > 0,
            F.round(
                (
                    F.col("mean_household_income_monthly_lkr")
                    / F.col("previous_mean_household_income_monthly_lkr")
                    - 1
                ) * 100,
                4
            )
        )
    )
    .withColumn(
        "median_household_income_change_percent",
        F.when(
            F.col("previous_median_household_income_monthly_lkr") > 0,
            F.round(
                (
                    F.col("median_household_income_monthly_lkr")
                    / F.col("previous_median_household_income_monthly_lkr")
                    - 1
                ) * 100,
                4
            )
        )
    )
    .withColumn(
        "mean_per_capita_income_change_percent",
        F.when(
            F.col("previous_mean_per_capita_income_monthly_lkr") > 0,
            F.round(
                (
                    F.col("mean_per_capita_income_monthly_lkr")
                    / F.col("previous_mean_per_capita_income_monthly_lkr")
                    - 1
                ) * 100,
                4
            )
        )
    )
    .withColumn(
        "mean_household_expenditure_change_percent",
        F.when(
            F.col("previous_mean_household_expenditure_monthly_lkr") > 0,
            F.round(
                (
                    F.col("mean_household_expenditure_monthly_lkr")
                    / F.col("previous_mean_household_expenditure_monthly_lkr")
                    - 1
                ) * 100,
                4
            )
        )
    )

    # Annualized compound growth across irregular survey gaps
    .withColumn(
        "mean_household_income_annualized_growth_percent",
        F.when(
            (F.col("previous_mean_household_income_monthly_lkr") > 0)
            & (F.col("years_since_previous_survey") > 0),
            F.round(
                (
                    F.exp(
                        F.log(
                            F.col("mean_household_income_monthly_lkr")
                            / F.col("previous_mean_household_income_monthly_lkr")
                        )
                        / F.col("years_since_previous_survey")
                    ) - 1
                ) * 100,
                4
            )
        )
    )
    .withColumn(
        "mean_household_expenditure_annualized_growth_percent",
        F.when(
            (F.col("previous_mean_household_expenditure_monthly_lkr") > 0)
            & (F.col("years_since_previous_survey") > 0),
            F.round(
                (
                    F.exp(
                        F.log(
                            F.col("mean_household_expenditure_monthly_lkr")
                            / F.col("previous_mean_household_expenditure_monthly_lkr")
                        )
                        / F.col("years_since_previous_survey")
                    ) - 1
                ) * 100,
                4
            )
        )
    )

    # Point changes for ratio and distribution indicators
    .withColumn(
        "food_ratio_change_pp",
        F.round(
            F.col("food_ratio_percent")
            - F.col("previous_food_ratio_percent"),
            4
        )
    )
    .withColumn(
        "income_gini_change_points",
        F.round(
            F.col("gini_household_income")
            - F.col("previous_gini_household_income"),
            4
        )
    )
    .withColumn(
        "poverty_headcount_change_pp",
        F.round(
            F.col("poverty_headcount_ratio_2002_based_percent")
            - F.col("previous_poverty_headcount_percent"),
            4
        )
    )

    .orderBy("representative_year")
)

cbsl_hies_change_features_df.select(
    "survey_period",
    "representative_year",
    "years_since_previous_survey",
    "mean_household_income_monthly_lkr",
    "mean_household_income_change_percent",
    "mean_household_income_annualized_growth_percent",
    "mean_household_expenditure_monthly_lkr",
    "mean_household_expenditure_change_percent",
    "mean_household_expenditure_annualized_growth_percent",
    "food_ratio_change_pp",
    "income_gini_change_points",
    "poverty_headcount_change_pp",
    "overlaps_cbsl_telecom_period"
).show(20, truncate=False)

In [0]:
from pyspark.sql import functions as F

print("HIES change-feature validation summary:")

cbsl_hies_change_features_df.agg(
    F.count("*").alias("row_count"),

    F.sum(
        F.col("mean_household_income_change_percent")
        .isNull().cast("int")
    ).alias("income_change_null_count"),

    F.sum(
        F.col("mean_household_income_annualized_growth_percent")
        .isNull().cast("int")
    ).alias("income_annualized_growth_null_count"),

    F.sum(
        F.col("mean_household_expenditure_change_percent")
        .isNull().cast("int")
    ).alias("expenditure_change_null_count"),

    F.sum(
        F.col("mean_household_expenditure_annualized_growth_percent")
        .isNull().cast("int")
    ).alias("expenditure_annualized_growth_null_count"),

    F.sum(
        F.col("food_ratio_change_pp")
        .isNull().cast("int")
    ).alias("food_ratio_change_null_count"),

    F.sum(
        F.col("income_gini_change_points")
        .isNull().cast("int")
    ).alias("income_gini_change_null_count"),

    F.sum(
        F.col("poverty_headcount_change_pp")
        .isNull().cast("int")
    ).alias("poverty_change_null_count"),

    F.min(
        "mean_household_income_annualized_growth_percent"
    ).alias("minimum_income_annualized_growth"),

    F.max(
        "mean_household_income_annualized_growth_percent"
    ).alias("maximum_income_annualized_growth"),

    F.min(
        "mean_household_expenditure_annualized_growth_percent"
    ).alias("minimum_expenditure_annualized_growth"),

    F.max(
        "mean_household_expenditure_annualized_growth_percent"
    ).alias("maximum_expenditure_annualized_growth")
).show(truncate=False)

print("Invalid-value checks:")

cbsl_hies_change_features_df.agg(
    F.sum(
        F.when(
            F.col("years_since_previous_survey") <= 0,
            1
        ).otherwise(0)
    ).alias("invalid_survey_gap_count"),

    F.sum(
        F.when(
            F.col("mean_household_income_annualized_growth_percent")
            <= -100,
            1
        ).otherwise(0)
    ).alias("invalid_income_growth_count"),

    F.sum(
        F.when(
            F.col("mean_household_expenditure_annualized_growth_percent")
            <= -100,
            1
        ).otherwise(0)
    ).alias("invalid_expenditure_growth_count")
).show(truncate=False)

In [0]:
cbsl_hies_analytical_df = (
    cbsl_hies_change_features_df
    .select(
        # Original survey metadata
        "survey_period",
        "survey_period_type",
        "period_start_year",
        "period_end_year",
        "representative_year",
        "previous_survey_period",
        "previous_representative_year",
        "years_since_previous_survey",

        # Alignment and interpretation metadata
        "alignment_method",
        "is_original_single_year_observation",
        "is_multi_year_survey_observation",
        "is_derived_annual_value",
        "overlaps_cbsl_telecom_period",

        # Income levels — nominal LKR
        "mean_household_income_monthly_lkr",
        "median_household_income_monthly_lkr",
        "mean_per_capita_income_monthly_lkr",

        # Expenditure levels — nominal LKR
        "mean_household_expenditure_monthly_lkr",
        "food_drink_expenditure_monthly_lkr",
        "non_food_expenditure_monthly_lkr",
        "food_ratio_percent",

        # Distribution, poverty, and household structure
        "gini_household_income",
        "poverty_headcount_ratio_2002_based_percent",
        "household_size",
        "income_receivers_per_household",

        # Period-to-period percentage changes
        "mean_household_income_change_percent",
        "median_household_income_change_percent",
        "mean_per_capita_income_change_percent",
        "mean_household_expenditure_change_percent",

        # Annualized nominal growth
        "mean_household_income_annualized_growth_percent",
        "mean_household_expenditure_annualized_growth_percent",

        # Ratio and distribution changes
        "food_ratio_change_pp",
        "income_gini_change_points",
        "poverty_headcount_change_pp"
    )
    .orderBy("representative_year")
)

print(f"Curated HIES rows: {cbsl_hies_analytical_df.count()}")
print(f"Curated HIES columns: {len(cbsl_hies_analytical_df.columns)}")

print("\nColumns:")
print(cbsl_hies_analytical_df.columns)

print("\nSchema:")
cbsl_hies_analytical_df.printSchema()

In [0]:
from pyspark.sql import functions as F

duplicate_hies_years_df = (
    cbsl_hies_analytical_df
    .groupBy("representative_year")
    .count()
    .filter(F.col("count") > 1)
)

print("Curated HIES final validation:")

cbsl_hies_analytical_df.agg(
    F.count("*").alias("row_count"),

    F.countDistinct("representative_year")
    .alias("distinct_representative_year_count"),

    F.min("representative_year")
    .alias("minimum_representative_year"),

    F.max("representative_year")
    .alias("maximum_representative_year"),

    F.sum(
        F.col("mean_household_income_monthly_lkr")
        .isNull().cast("int")
    ).alias("mean_income_null_count"),

    F.sum(
        F.col("mean_household_expenditure_monthly_lkr")
        .isNull().cast("int")
    ).alias("expenditure_null_count"),

    F.sum(
        F.col("food_ratio_percent")
        .isNull().cast("int")
    ).alias("food_ratio_null_count"),

    F.sum(
        F.col("alignment_method")
        .isNull().cast("int")
    ).alias("alignment_method_null_count"),

    F.sum(
        F.col("overlaps_cbsl_telecom_period")
        .cast("int")
    ).alias("telecom_overlap_count")
).show(truncate=False)

print(f"Duplicate representative-year count: {duplicate_hies_years_df.count()}")

duplicate_hies_years_df.orderBy(
    "representative_year"
).show(truncate=False)

print("Expected change-feature null counts:")

cbsl_hies_analytical_df.agg(
    F.sum(
        F.col("mean_household_income_change_percent")
        .isNull().cast("int")
    ).alias("income_change_null_count"),

    F.sum(
        F.col("mean_household_income_annualized_growth_percent")
        .isNull().cast("int")
    ).alias("income_annualized_growth_null_count"),

    F.sum(
        F.col("poverty_headcount_change_pp")
        .isNull().cast("int")
    ).alias("poverty_change_null_count")
).show(truncate=False)

In [0]:
from pyspark.sql import functions as F

cbsl_poverty_silver_df = spark.table(
    "workspace.census360.silver_cbsl_poverty_indicators"
)

poverty_national_source_df = (
    cbsl_poverty_silver_df
    .filter(F.col("geography_type") == "national")
)

poverty_national_metadata_df = (
    poverty_national_source_df
    .select(
        "geography_type",
        "geography_name",
        "survey_period",
        "survey_period_type",
        "period_start_year",
        "period_end_year",
        "definition_variant"
    )
    .distinct()
)

poverty_national_values_wide_df = (
    poverty_national_source_df
    .groupBy(
        "geography_type",
        "geography_name",
        "survey_period",
        "definition_variant"
    )
    .pivot("metric_code")
    .agg(F.first("value_numeric"))
)

cbsl_poverty_national_features_df = (
    poverty_national_metadata_df
    .join(
        poverty_national_values_wide_df,
        on=[
            "geography_type",
            "geography_name",
            "survey_period",
            "definition_variant"
        ],
        how="inner"
    )
    .orderBy(
        "period_end_year",
        "definition_variant"
    )
)

print(
    f"National poverty rows: "
    f"{cbsl_poverty_national_features_df.count()}"
)
print(
    f"National poverty columns: "
    f"{len(cbsl_poverty_national_features_df.columns)}"
)

print("\nColumns:")
print(cbsl_poverty_national_features_df.columns)

print("\nNational poverty feature data:")
cbsl_poverty_national_features_df.show(
    20,
    truncate=False
)

In [0]:
from pyspark.sql import functions as F

duplicate_national_poverty_keys_df = (
    cbsl_poverty_national_features_df
    .groupBy(
        "geography_type",
        "geography_name",
        "survey_period",
        "definition_variant"
    )
    .count()
    .filter(F.col("count") > 1)
)

print("National poverty validation summary:")

cbsl_poverty_national_features_df.agg(
    F.count("*").alias("row_count"),

    F.countDistinct("survey_period")
    .alias("distinct_survey_period_count"),

    F.countDistinct("definition_variant")
    .alias("distinct_definition_variant_count"),

    F.sum(
        F.col("poor_household_percentage")
        .isNull().cast("int")
    ).alias("poor_household_null_count"),

    F.sum(
        F.col("poverty_gap_index_percent")
        .isNull().cast("int")
    ).alias("poverty_gap_null_count"),

    F.sum(
        F.col("poverty_headcount_index_percent")
        .isNull().cast("int")
    ).alias("poverty_headcount_null_count"),

    F.min("poverty_headcount_index_percent")
    .alias("minimum_poverty_headcount"),

    F.max("poverty_headcount_index_percent")
    .alias("maximum_poverty_headcount")
).show(truncate=False)

print(
    f"Duplicate national poverty key count: "
    f"{duplicate_national_poverty_keys_df.count()}"
)

duplicate_national_poverty_keys_df.show(
    truncate=False
)

print("2019 definition comparison:")

cbsl_poverty_national_features_df.filter(
    F.col("period_end_year") == 2019
).select(
    "survey_period",
    "definition_variant",
    "poor_household_percentage",
    "poverty_gap_index_percent",
    "poverty_headcount_index_percent"
).orderBy(
    "definition_variant"
).show(truncate=False)

In [0]:
from pyspark.sql import functions as F

cbsl_poverty_national_aligned_df = (
    cbsl_poverty_national_features_df

    .withColumn(
        "representative_year",
        F.col("period_end_year")
    )

    .withColumn(
        "alignment_method",
        F.lit("survey_period_end_year")
    )

    .withColumn(
        "is_multi_year_survey_observation",
        F.col("survey_period_type") == "multi_year"
    )

    .withColumn(
        "is_derived_annual_value",
        F.lit(False)
    )

    .withColumn(
        "overlaps_cbsl_telecom_period",
        F.col("representative_year").between(2013, 2024)
    )

    # Definition grouping for safe comparisons
    .withColumn(
        "poverty_definition_group",
        F.when(
            F.col("definition_variant") == "standard",
            F.lit("standard_series")
        )
        .when(
            F.col("definition_variant") == "old_poverty_line_2002_ccpi",
            F.lit("old_2002_ccpi_series")
        )
        .when(
            F.col("definition_variant") == "updated_poverty_line_2012_13_ncpi",
            F.lit("updated_2012_13_ncpi_series")
        )
        .otherwise(F.lit("unknown_definition"))
    )

    .withColumn(
        "safe_for_same_definition_change",
        F.col("definition_variant") == "standard"
    )

    .orderBy(
        "representative_year",
        "definition_variant"
    )
)

print("National poverty alignment and comparability metadata:")

cbsl_poverty_national_aligned_df.select(
    "survey_period",
    "representative_year",
    "definition_variant",
    "poverty_definition_group",
    "alignment_method",
    "is_multi_year_survey_observation",
    "is_derived_annual_value",
    "overlaps_cbsl_telecom_period",
    "safe_for_same_definition_change",
    "poverty_headcount_index_percent"
).show(20, truncate=False)

print("Comparability summary:")

cbsl_poverty_national_aligned_df.groupBy(
    "poverty_definition_group",
    "safe_for_same_definition_change"
).agg(
    F.count("*").alias("observation_count"),
    F.min("representative_year").alias("minimum_year"),
    F.max("representative_year").alias("maximum_year")
).orderBy(
    "poverty_definition_group"
).show(truncate=False)

In [0]:
from pyspark.sql import functions as F

cbsl_poverty_national_aligned_v2_df = (
    cbsl_poverty_national_aligned_df

    # Comparable historical series:
    # 2012/13 standard, 2016 standard, and 2019(a) old poverty line
    .withColumn(
        "comparison_series",
        F.when(
            F.col("definition_variant").isin(
                "standard",
                "old_poverty_line_2002_ccpi"
            ),
            F.lit("historical_old_line_comparable_series")
        )
        .when(
            F.col("definition_variant")
            == "updated_poverty_line_2012_13_ncpi",
            F.lit("updated_2012_13_ncpi_series")
        )
        .otherwise(
            F.lit("unknown_series")
        )
    )

    .withColumn(
        "safe_for_historical_change_analysis",
        F.col("definition_variant").isin(
            "standard",
            "old_poverty_line_2002_ccpi"
        )
    )

    .withColumn(
        "requires_definition_warning",
        F.col("definition_variant")
        == "updated_poverty_line_2012_13_ncpi"
    )

    .orderBy(
        "representative_year",
        "definition_variant"
    )
)

print("Refined national poverty comparison groups:")

cbsl_poverty_national_aligned_v2_df.select(
    "survey_period",
    "representative_year",
    "definition_variant",
    "comparison_series",
    "safe_for_historical_change_analysis",
    "requires_definition_warning",
    "poverty_headcount_index_percent",
    "poor_household_percentage",
    "poverty_gap_index_percent"
).show(20, truncate=False)

print("Refined comparison-series summary:")

cbsl_poverty_national_aligned_v2_df.groupBy(
    "comparison_series",
    "safe_for_historical_change_analysis"
).agg(
    F.count("*").alias("observation_count"),
    F.min("representative_year").alias("minimum_year"),
    F.max("representative_year").alias("maximum_year")
).orderBy(
    "comparison_series"
).show(truncate=False)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

poverty_change_window = (
    Window
    .partitionBy("comparison_series")
    .orderBy("representative_year")
)

cbsl_poverty_national_change_features_df = (
    cbsl_poverty_national_aligned_v2_df

    # Previous observation metadata
    .withColumn(
        "previous_survey_period",
        F.lag("survey_period", 1).over(poverty_change_window)
    )
    .withColumn(
        "previous_representative_year",
        F.lag("representative_year", 1).over(poverty_change_window)
    )
    .withColumn(
        "previous_definition_variant",
        F.lag("definition_variant", 1).over(poverty_change_window)
    )
    .withColumn(
        "years_since_previous_observation",
        F.when(
            F.col("previous_representative_year").isNotNull(),
            F.col("representative_year")
            - F.col("previous_representative_year")
        )
    )

    # Previous poverty values
    .withColumn(
        "previous_poverty_headcount_index_percent",
        F.lag("poverty_headcount_index_percent", 1)
        .over(poverty_change_window)
    )
    .withColumn(
        "previous_poor_household_percentage",
        F.lag("poor_household_percentage", 1)
        .over(poverty_change_window)
    )
    .withColumn(
        "previous_poverty_gap_index_percent",
        F.lag("poverty_gap_index_percent", 1)
        .over(poverty_change_window)
    )

    # Point changes
    .withColumn(
        "poverty_headcount_change_pp",
        F.when(
            F.col("safe_for_historical_change_analysis"),
            F.round(
                F.col("poverty_headcount_index_percent")
                - F.col("previous_poverty_headcount_index_percent"),
                4
            )
        )
    )
    .withColumn(
        "poor_household_change_pp",
        F.when(
            F.col("safe_for_historical_change_analysis"),
            F.round(
                F.col("poor_household_percentage")
                - F.col("previous_poor_household_percentage"),
                4
            )
        )
    )
    .withColumn(
        "poverty_gap_change_pp",
        F.when(
            F.col("safe_for_historical_change_analysis"),
            F.round(
                F.col("poverty_gap_index_percent")
                - F.col("previous_poverty_gap_index_percent"),
                4
            )
        )
    )

    # Explicit transition label
    .withColumn(
        "comparison_transition",
        F.when(
            F.col("previous_survey_period").isNotNull(),
            F.concat_ws(
                " to ",
                F.col("previous_survey_period"),
                F.col("survey_period")
            )
        )
    )

    .orderBy(
        "representative_year",
        "definition_variant"
    )
)

print("National poverty change features:")

cbsl_poverty_national_change_features_df.select(
    "survey_period",
    "representative_year",
    "definition_variant",
    "comparison_series",
    "comparison_transition",
    "previous_definition_variant",
    "years_since_previous_observation",
    "poverty_headcount_index_percent",
    "poverty_headcount_change_pp",
    "poor_household_percentage",
    "poor_household_change_pp",
    "poverty_gap_index_percent",
    "poverty_gap_change_pp",
    "requires_definition_warning"
).show(20, truncate=False)

In [0]:
from pyspark.sql import functions as F

print("National poverty change-feature validation:")

cbsl_poverty_national_change_features_df.agg(
    F.count("*").alias("row_count"),

    F.sum(
        F.col("poverty_headcount_change_pp")
        .isNull().cast("int")
    ).alias("poverty_headcount_change_null_count"),

    F.sum(
        F.col("poor_household_change_pp")
        .isNull().cast("int")
    ).alias("poor_household_change_null_count"),

    F.sum(
        F.col("poverty_gap_change_pp")
        .isNull().cast("int")
    ).alias("poverty_gap_change_null_count"),

    F.sum(
        F.when(
            F.col("requires_definition_warning")
            & F.col("poverty_headcount_change_pp").isNotNull(),
            1
        ).otherwise(0)
    ).alias("invalid_updated_definition_change_count"),

    F.sum(
        F.when(
            F.col("years_since_previous_observation") <= 0,
            1
        ).otherwise(0)
    ).alias("invalid_observation_gap_count"),

    F.min("poverty_headcount_change_pp")
    .alias("minimum_poverty_headcount_change_pp"),

    F.max("poverty_headcount_change_pp")
    .alias("maximum_poverty_headcount_change_pp")
).show(truncate=False)

print("Validated change transitions:")

cbsl_poverty_national_change_features_df.filter(
    F.col("comparison_transition").isNotNull()
).select(
    "comparison_transition",
    "previous_definition_variant",
    "definition_variant",
    "years_since_previous_observation",
    "poverty_headcount_change_pp",
    "poor_household_change_pp",
    "poverty_gap_change_pp"
).orderBy(
    "representative_year"
).show(truncate=False)

In [0]:
cbsl_poverty_national_analytical_df = (
    cbsl_poverty_national_change_features_df
    .select(
        # Geography and survey metadata
        "geography_type",
        "geography_name",
        "survey_period",
        "survey_period_type",
        "period_start_year",
        "period_end_year",
        "representative_year",

        # Poverty-definition metadata
        "definition_variant",
        "poverty_definition_group",
        "comparison_series",
        "alignment_method",
        "is_multi_year_survey_observation",
        "is_derived_annual_value",
        "overlaps_cbsl_telecom_period",
        "safe_for_historical_change_analysis",
        "requires_definition_warning",

        # Poverty levels
        "poverty_headcount_index_percent",
        "poor_household_percentage",
        "poverty_gap_index_percent",

        # Previous-observation metadata
        "previous_survey_period",
        "previous_representative_year",
        "previous_definition_variant",
        "years_since_previous_observation",
        "comparison_transition",

        # Safe change features
        "poverty_headcount_change_pp",
        "poor_household_change_pp",
        "poverty_gap_change_pp"
    )
    .orderBy(
        "representative_year",
        "definition_variant"
    )
)

print(
    f"Curated national poverty rows: "
    f"{cbsl_poverty_national_analytical_df.count()}"
)

print(
    f"Curated national poverty columns: "
    f"{len(cbsl_poverty_national_analytical_df.columns)}"
)

print("\nColumns:")
print(cbsl_poverty_national_analytical_df.columns)

print("\nSchema:")
cbsl_poverty_national_analytical_df.printSchema()

In [0]:
from pyspark.sql import functions as F

duplicate_national_poverty_rows_df = (
    cbsl_poverty_national_analytical_df
    .groupBy(
        "geography_type",
        "geography_name",
        "survey_period",
        "definition_variant"
    )
    .count()
    .filter(F.col("count") > 1)
)

print("Curated national poverty final validation:")

cbsl_poverty_national_analytical_df.agg(
    F.count("*").alias("row_count"),

    F.countDistinct("survey_period")
    .alias("distinct_survey_period_count"),

    F.countDistinct("definition_variant")
    .alias("distinct_definition_variant_count"),

    F.sum(
        F.col("poverty_headcount_index_percent")
        .isNull().cast("int")
    ).alias("poverty_headcount_null_count"),

    F.sum(
        F.col("poor_household_percentage")
        .isNull().cast("int")
    ).alias("poor_household_null_count"),

    F.sum(
        F.col("poverty_gap_index_percent")
        .isNull().cast("int")
    ).alias("poverty_gap_null_count"),

    F.sum(
        F.col("comparison_series")
        .isNull().cast("int")
    ).alias("comparison_series_null_count"),

    F.sum(
        F.col("overlaps_cbsl_telecom_period")
        .cast("int")
    ).alias("telecom_overlap_count"),

    F.sum(
        F.col("requires_definition_warning")
        .cast("int")
    ).alias("definition_warning_count")
).show(truncate=False)

print(
    f"Duplicate analytical key count: "
    f"{duplicate_national_poverty_rows_df.count()}"
)

duplicate_national_poverty_rows_df.show(truncate=False)

print("Expected change-feature null counts:")

cbsl_poverty_national_analytical_df.agg(
    F.sum(
        F.col("poverty_headcount_change_pp")
        .isNull().cast("int")
    ).alias("poverty_headcount_change_null_count"),

    F.sum(
        F.col("poor_household_change_pp")
        .isNull().cast("int")
    ).alias("poor_household_change_null_count"),

    F.sum(
        F.col("poverty_gap_change_pp")
        .isNull().cast("int")
    ).alias("poverty_gap_change_null_count")
).show(truncate=False)

In [0]:
from pyspark.sql import functions as F

cbsl_poverty_silver_df = spark.table(
    "workspace.census360.silver_cbsl_poverty_indicators"
)

poverty_province_source_df = (
    cbsl_poverty_silver_df
    .filter(F.col("geography_type") == "province")
)

poverty_province_metadata_df = (
    poverty_province_source_df
    .select(
        "geography_type",
        "geography_name",
        "survey_period",
        "survey_period_type",
        "period_start_year",
        "period_end_year",
        "definition_variant"
    )
    .distinct()
)

poverty_province_values_wide_df = (
    poverty_province_source_df
    .groupBy(
        "geography_type",
        "geography_name",
        "survey_period",
        "definition_variant"
    )
    .pivot("metric_code")
    .agg(F.first("value_numeric"))
)

cbsl_poverty_province_features_df = (
    poverty_province_metadata_df
    .join(
        poverty_province_values_wide_df,
        on=[
            "geography_type",
            "geography_name",
            "survey_period",
            "definition_variant"
        ],
        how="inner"
    )
    .orderBy(
        "geography_name",
        "period_end_year",
        "definition_variant"
    )
)

print(
    f"Province poverty rows: "
    f"{cbsl_poverty_province_features_df.count()}"
)

print(
    f"Province poverty columns: "
    f"{len(cbsl_poverty_province_features_df.columns)}"
)

print("\nColumns:")
print(cbsl_poverty_province_features_df.columns)

print("\nProvince poverty feature data:")

cbsl_poverty_province_features_df.show(
    50,
    truncate=False
)

In [0]:
from pyspark.sql import functions as F

duplicate_province_poverty_keys_df = (
    cbsl_poverty_province_features_df
    .groupBy(
        "geography_type",
        "geography_name",
        "survey_period",
        "definition_variant"
    )
    .count()
    .filter(F.col("count") > 1)
)

print("Province poverty validation summary:")

cbsl_poverty_province_features_df.agg(
    F.count("*").alias("row_count"),

    F.countDistinct("geography_name")
    .alias("province_count"),

    F.countDistinct("survey_period")
    .alias("survey_period_count"),

    F.countDistinct("definition_variant")
    .alias("definition_variant_count"),

    F.sum(
        F.col("poor_household_percentage")
        .isNull().cast("int")
    ).alias("poor_household_null_count"),

    F.sum(
        F.col("poverty_gap_index_percent")
        .isNull().cast("int")
    ).alias("poverty_gap_null_count"),

    F.sum(
        F.col("poverty_headcount_index_percent")
        .isNull().cast("int")
    ).alias("poverty_headcount_null_count")
).show(truncate=False)

print("Rows per province:")

cbsl_poverty_province_features_df.groupBy(
    "geography_name"
).count().orderBy(
    "geography_name"
).show(20, truncate=False)

print(
    f"Duplicate province poverty key count: "
    f"{duplicate_province_poverty_keys_df.count()}"
)

duplicate_province_poverty_keys_df.show(truncate=False)

print("Definition coverage by survey period:")

cbsl_poverty_province_features_df.groupBy(
    "survey_period",
    "definition_variant"
).agg(
    F.countDistinct("geography_name").alias("province_count"),
    F.count("*").alias("row_count")
).orderBy(
    "period_end_year",
    "definition_variant"
).show(truncate=False)

In [0]:
print("Definition coverage by survey period:")

cbsl_poverty_province_features_df.groupBy(
    "survey_period",
    "period_end_year",
    "definition_variant"
).agg(
    F.countDistinct("geography_name").alias("province_count"),
    F.count("*").alias("row_count")
).orderBy(
    "period_end_year",
    "definition_variant"
).show(truncate=False)

In [0]:
from pyspark.sql import functions as F

duplicate_province_poverty_keys_df = (
    cbsl_poverty_province_features_df
    .groupBy(
        "geography_type",
        "geography_name",
        "survey_period",
        "definition_variant"
    )
    .count()
    .filter(F.col("count") > 1)
)

print("Province poverty validation summary:")

cbsl_poverty_province_features_df.agg(
    F.count("*").alias("row_count"),
    F.countDistinct("geography_name").alias("province_count"),
    F.countDistinct("survey_period").alias("survey_period_count"),
    F.countDistinct("definition_variant").alias("definition_variant_count"),

    F.sum(
        F.col("poor_household_percentage").isNull().cast("int")
    ).alias("poor_household_null_count"),

    F.sum(
        F.col("poverty_gap_index_percent").isNull().cast("int")
    ).alias("poverty_gap_null_count"),

    F.sum(
        F.col("poverty_headcount_index_percent").isNull().cast("int")
    ).alias("poverty_headcount_null_count")
).show(truncate=False)

print("Rows per province:")

cbsl_poverty_province_features_df.groupBy(
    "geography_name"
).count().orderBy(
    "geography_name"
).show(20, truncate=False)

print(
    f"Duplicate province poverty key count: "
    f"{duplicate_province_poverty_keys_df.count()}"
)

duplicate_province_poverty_keys_df.show(truncate=False)

In [0]:
from pyspark.sql import functions as F

cbsl_poverty_province_aligned_df = (
    cbsl_poverty_province_features_df

    .withColumn(
        "representative_year",
        F.col("period_end_year")
    )

    .withColumn(
        "alignment_method",
        F.lit("survey_period_end_year")
    )

    .withColumn(
        "is_multi_year_survey_observation",
        F.col("survey_period_type") == "multi_year"
    )

    .withColumn(
        "is_derived_annual_value",
        F.lit(False)
    )

    .withColumn(
        "overlaps_cbsl_telecom_period",
        F.col("representative_year").between(2013, 2024)
    )

    .withColumn(
        "comparison_series",
        F.when(
            F.col("definition_variant").isin(
                "standard",
                "old_poverty_line_2002_ccpi"
            ),
            F.lit("historical_old_line_comparable_series")
        )
        .when(
            F.col("definition_variant")
            == "updated_poverty_line_2012_13_ncpi",
            F.lit("updated_2012_13_ncpi_series")
        )
        .otherwise(
            F.lit("unknown_series")
        )
    )

    .withColumn(
        "safe_for_historical_change_analysis",
        F.col("definition_variant").isin(
            "standard",
            "old_poverty_line_2002_ccpi"
        )
    )

    .withColumn(
        "requires_definition_warning",
        F.col("definition_variant")
        == "updated_poverty_line_2012_13_ncpi"
    )

    .orderBy(
        "geography_name",
        "representative_year",
        "definition_variant"
    )
)

print("Province poverty alignment and comparison metadata:")

cbsl_poverty_province_aligned_df.select(
    "geography_name",
    "survey_period",
    "representative_year",
    "definition_variant",
    "comparison_series",
    "safe_for_historical_change_analysis",
    "requires_definition_warning",
    "poverty_headcount_index_percent"
).show(50, truncate=False)

print("Comparison-series coverage:")

cbsl_poverty_province_aligned_df.groupBy(
    "comparison_series",
    "safe_for_historical_change_analysis"
).agg(
    F.countDistinct("geography_name").alias("province_count"),
    F.count("*").alias("observation_count"),
    F.min("representative_year").alias("minimum_year"),
    F.max("representative_year").alias("maximum_year")
).orderBy(
    "comparison_series"
).show(truncate=False)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

province_poverty_change_window = (
    Window
    .partitionBy(
        "geography_name",
        "comparison_series"
    )
    .orderBy("representative_year")
)

cbsl_poverty_province_change_features_df = (
    cbsl_poverty_province_aligned_df

    # Previous observation metadata within each province
    .withColumn(
        "previous_survey_period",
        F.lag("survey_period", 1)
        .over(province_poverty_change_window)
    )
    .withColumn(
        "previous_representative_year",
        F.lag("representative_year", 1)
        .over(province_poverty_change_window)
    )
    .withColumn(
        "previous_definition_variant",
        F.lag("definition_variant", 1)
        .over(province_poverty_change_window)
    )
    .withColumn(
        "years_since_previous_observation",
        F.when(
            F.col("previous_representative_year").isNotNull(),
            F.col("representative_year")
            - F.col("previous_representative_year")
        )
    )

    # Previous poverty values
    .withColumn(
        "previous_poverty_headcount_index_percent",
        F.lag("poverty_headcount_index_percent", 1)
        .over(province_poverty_change_window)
    )
    .withColumn(
        "previous_poor_household_percentage",
        F.lag("poor_household_percentage", 1)
        .over(province_poverty_change_window)
    )
    .withColumn(
        "previous_poverty_gap_index_percent",
        F.lag("poverty_gap_index_percent", 1)
        .over(province_poverty_change_window)
    )

    # Safe point changes
    .withColumn(
        "poverty_headcount_change_pp",
        F.when(
            F.col("safe_for_historical_change_analysis"),
            F.round(
                F.col("poverty_headcount_index_percent")
                - F.col("previous_poverty_headcount_index_percent"),
                4
            )
        )
    )
    .withColumn(
        "poor_household_change_pp",
        F.when(
            F.col("safe_for_historical_change_analysis"),
            F.round(
                F.col("poor_household_percentage")
                - F.col("previous_poor_household_percentage"),
                4
            )
        )
    )
    .withColumn(
        "poverty_gap_change_pp",
        F.when(
            F.col("safe_for_historical_change_analysis"),
            F.round(
                F.col("poverty_gap_index_percent")
                - F.col("previous_poverty_gap_index_percent"),
                4
            )
        )
    )

    # Transition label
    .withColumn(
        "comparison_transition",
        F.when(
            F.col("previous_survey_period").isNotNull(),
            F.concat_ws(
                " to ",
                F.col("previous_survey_period"),
                F.col("survey_period")
            )
        )
    )

    .orderBy(
        "geography_name",
        "representative_year",
        "definition_variant"
    )
)

print("Province poverty change features:")

cbsl_poverty_province_change_features_df.select(
    "geography_name",
    "survey_period",
    "representative_year",
    "definition_variant",
    "comparison_transition",
    "years_since_previous_observation",
    "poverty_headcount_index_percent",
    "poverty_headcount_change_pp",
    "poor_household_change_pp",
    "poverty_gap_change_pp",
    "requires_definition_warning"
).show(50, truncate=False)

In [0]:
from pyspark.sql import functions as F

print("Province poverty change-feature validation:")

cbsl_poverty_province_change_features_df.agg(
    F.count("*").alias("row_count"),

    F.sum(
        F.col("poverty_headcount_change_pp").isNull().cast("int")
    ).alias("poverty_headcount_change_null_count"),

    F.sum(
        F.col("poor_household_change_pp").isNull().cast("int")
    ).alias("poor_household_change_null_count"),

    F.sum(
        F.col("poverty_gap_change_pp").isNull().cast("int")
    ).alias("poverty_gap_change_null_count"),

    F.sum(
        F.when(
            F.col("requires_definition_warning")
            & F.col("poverty_headcount_change_pp").isNotNull(),
            1
        ).otherwise(0)
    ).alias("invalid_updated_definition_change_count"),

    F.sum(
        F.when(
            F.col("years_since_previous_observation") <= 0,
            1
        ).otherwise(0)
    ).alias("invalid_observation_gap_count"),

    F.sum(
        F.when(
            F.col("poverty_headcount_change_pp") > 0,
            1
        ).otherwise(0)
    ).alias("poverty_headcount_increase_count"),

    F.sum(
        F.when(
            F.col("poverty_headcount_change_pp") < 0,
            1
        ).otherwise(0)
    ).alias("poverty_headcount_decrease_count")
).show(truncate=False)

print("Valid transition coverage:")

cbsl_poverty_province_change_features_df.filter(
    F.col("comparison_transition").isNotNull()
).groupBy(
    "comparison_transition"
).agg(
    F.countDistinct("geography_name").alias("province_count"),
    F.count("*").alias("row_count"),
    F.min("poverty_headcount_change_pp").alias("minimum_change_pp"),
    F.max("poverty_headcount_change_pp").alias("maximum_change_pp")
).orderBy(
    "comparison_transition"
).show(truncate=False)

In [0]:
cbsl_poverty_province_analytical_df = (
    cbsl_poverty_province_change_features_df
    .select(
        # Geography and survey metadata
        "geography_type",
        "geography_name",
        "survey_period",
        "survey_period_type",
        "period_start_year",
        "period_end_year",
        "representative_year",

        # Definition and comparison metadata
        "definition_variant",
        "comparison_series",
        "alignment_method",
        "is_multi_year_survey_observation",
        "is_derived_annual_value",
        "overlaps_cbsl_telecom_period",
        "safe_for_historical_change_analysis",
        "requires_definition_warning",

        # Poverty levels
        "poverty_headcount_index_percent",
        "poor_household_percentage",
        "poverty_gap_index_percent",

        # Previous-observation metadata
        "previous_survey_period",
        "previous_representative_year",
        "previous_definition_variant",
        "years_since_previous_observation",
        "comparison_transition",

        # Safe change features
        "poverty_headcount_change_pp",
        "poor_household_change_pp",
        "poverty_gap_change_pp"
    )
    .orderBy(
        "geography_name",
        "representative_year",
        "definition_variant"
    )
)

print(
    f"Curated province poverty rows: "
    f"{cbsl_poverty_province_analytical_df.count()}"
)

print(
    f"Curated province poverty columns: "
    f"{len(cbsl_poverty_province_analytical_df.columns)}"
)

print("\nColumns:")
print(cbsl_poverty_province_analytical_df.columns)

print("\nSchema:")
cbsl_poverty_province_analytical_df.printSchema()

In [0]:
from pyspark.sql import functions as F

duplicate_province_poverty_rows_df = (
    cbsl_poverty_province_analytical_df
    .groupBy(
        "geography_type",
        "geography_name",
        "survey_period",
        "definition_variant"
    )
    .count()
    .filter(F.col("count") > 1)
)

print("Curated province poverty final validation:")

cbsl_poverty_province_analytical_df.agg(
    F.count("*").alias("row_count"),

    F.countDistinct("geography_name")
    .alias("province_count"),

    F.countDistinct("survey_period")
    .alias("survey_period_count"),

    F.sum(
        F.col("poverty_headcount_index_percent")
        .isNull().cast("int")
    ).alias("poverty_headcount_null_count"),

    F.sum(
        F.col("poor_household_percentage")
        .isNull().cast("int")
    ).alias("poor_household_null_count"),

    F.sum(
        F.col("poverty_gap_index_percent")
        .isNull().cast("int")
    ).alias("poverty_gap_null_count"),

    F.sum(
        F.col("requires_definition_warning")
        .cast("int")
    ).alias("definition_warning_count"),

    F.sum(
        F.col("poverty_headcount_change_pp")
        .isNotNull().cast("int")
    ).alias("valid_change_row_count")
).show(truncate=False)

print(
    f"Duplicate analytical key count: "
    f"{duplicate_province_poverty_rows_df.count()}"
)

duplicate_province_poverty_rows_df.show(truncate=False)

print("Records per province and change availability:")

cbsl_poverty_province_analytical_df.groupBy(
    "geography_name"
).agg(
    F.count("*").alias("row_count"),

    F.sum(
        F.col("requires_definition_warning").cast("int")
    ).alias("definition_warning_count"),

    F.sum(
        F.col("poverty_headcount_change_pp")
        .isNotNull().cast("int")
    ).alias("valid_change_count")
).orderBy(
    "geography_name"
).show(20, truncate=False)

In [0]:
from pyspark.sql import functions as F

cbsl_provincial_silver_df = spark.table(
    "workspace.census360.silver_cbsl_provincial_socioeconomic"
)

print("Provincial socioeconomic metric catalogue:")

cbsl_provincial_silver_df.select(
    "section_name",
    "subsection_name",
    "metric_position",
    "metric_code",
    "metric_label",
    "unit"
).distinct().orderBy(
    "section_name",
    "subsection_name",
    "metric_position"
).show(100, truncate=False)

print("Metric count by section:")

cbsl_provincial_silver_df.groupBy(
    "section_name"
).agg(
    F.countDistinct("metric_code").alias("metric_count")
).orderBy(
    "section_name"
).show(truncate=False)

In [0]:
from pyspark.sql import functions as F

provincial_candidate_metric_codes = [
    "individuals_per_household",
    "income_receivers_per_household",
    "mean_income_per_household_monthly_lkr",
    "mean_income_per_person_monthly_lkr",
    "median_income_per_household_monthly_lkr",
    "gini_coefficient_households",
    "household_expenditure_monthly_lkr",
    "expenditure_share_communication_percent",
    "expenditure_share_education_percent",
    "expenditure_share_transport_percent",
    "expenditure_share_housing_percent",
    "education_passed_gce_ol_percent",
    "education_passed_gce_al_and_above_percent"
]

available_candidate_metrics_df = (
    cbsl_provincial_silver_df
    .filter(
        F.col("metric_code").isin(provincial_candidate_metric_codes)
    )
    .select(
        "section_name",
        "subsection_name",
        "metric_code",
        "metric_label",
        "unit"
    )
    .distinct()
    .orderBy("section_name", "metric_code")
)

available_metric_codes = {
    row["metric_code"]
    for row in available_candidate_metrics_df.select("metric_code").collect()
}

missing_metric_codes = sorted(
    set(provincial_candidate_metric_codes) - available_metric_codes
)

print("Validated provincial candidate metrics:")
available_candidate_metrics_df.show(30, truncate=False)

print(f"Requested candidate metric count: {len(provincial_candidate_metric_codes)}")
print(f"Available candidate metric count: {len(available_metric_codes)}")
print(f"Missing candidate metric count: {len(missing_metric_codes)}")
print(f"Missing metric codes: {missing_metric_codes}")

In [0]:
from pyspark.sql import functions as F

cbsl_provincial_candidate_long_df = (
    cbsl_provincial_silver_df
    .filter(
        F.col("metric_code").isin(provincial_candidate_metric_codes)
    )
)

duplicate_provincial_candidate_keys_df = (
    cbsl_provincial_candidate_long_df
    .groupBy(
        "analysis_year",
        "geography_type",
        "geography_name",
        "metric_code"
    )
    .count()
    .filter(F.col("count") > 1)
)

print("Provincial candidate metric validation:")

cbsl_provincial_candidate_long_df.agg(
    F.count("*").alias("row_count"),
    F.countDistinct("metric_code").alias("metric_count"),
    F.countDistinct("analysis_year").alias("year_count"),
    F.countDistinct("geography_name").alias("geography_count"),

    F.sum(
        F.col("value_numeric").isNull().cast("int")
    ).alias("numeric_null_count")
).show(truncate=False)

print("Coverage by geography type and year:")

cbsl_provincial_candidate_long_df.groupBy(
    "geography_type",
    "analysis_year"
).agg(
    F.countDistinct("geography_name").alias("geography_count"),
    F.countDistinct("metric_code").alias("metric_count"),
    F.count("*").alias("row_count")
).orderBy(
    "geography_type",
    "analysis_year"
).show(truncate=False)

print(
    f"Duplicate candidate analytical key count: "
    f"{duplicate_provincial_candidate_keys_df.count()}"
)

duplicate_provincial_candidate_keys_df.show(
    20,
    truncate=False
)

In [0]:
from pyspark.sql import functions as F

cbsl_provincial_features_df = (
    cbsl_provincial_candidate_long_df
    .filter(F.col("geography_type") == "province")
    .groupBy(
        "analysis_year",
        "geography_type",
        "geography_name"
    )
    .pivot(
        "metric_code",
        provincial_candidate_metric_codes
    )
    .agg(F.first("value_numeric"))
    .orderBy(
        "geography_name",
        "analysis_year"
    )
)

print(
    f"Province socioeconomic feature rows: "
    f"{cbsl_provincial_features_df.count()}"
)

print(
    f"Province socioeconomic feature columns: "
    f"{len(cbsl_provincial_features_df.columns)}"
)

print("\nColumns:")
print(cbsl_provincial_features_df.columns)

print("\nProvince-year socioeconomic features:")

cbsl_provincial_features_df.show(
    30,
    truncate=False
)

In [0]:
from pyspark.sql import functions as F

duplicate_province_year_keys_df = (
    cbsl_provincial_features_df
    .groupBy(
        "analysis_year",
        "geography_type",
        "geography_name"
    )
    .count()
    .filter(F.col("count") > 1)
)

feature_columns = [
    column_name
    for column_name in cbsl_provincial_features_df.columns
    if column_name not in {
        "analysis_year",
        "geography_type",
        "geography_name"
    }
]

null_summary_expressions = [
    F.sum(
        F.col(column_name).isNull().cast("int")
    ).alias(f"{column_name}_null_count")
    for column_name in feature_columns
]

print("Province-year feature validation summary:")

cbsl_provincial_features_df.agg(
    F.count("*").alias("row_count"),
    F.countDistinct("geography_name").alias("province_count"),
    F.countDistinct("analysis_year").alias("year_count"),
    *null_summary_expressions
).show(truncate=False)

print("Rows per province:")

cbsl_provincial_features_df.groupBy(
    "geography_name"
).agg(
    F.count("*").alias("row_count"),
    F.countDistinct("analysis_year").alias("year_count")
).orderBy(
    "geography_name"
).show(20, truncate=False)

print("Rows per year:")

cbsl_provincial_features_df.groupBy(
    "analysis_year"
).agg(
    F.countDistinct("geography_name").alias("province_count"),
    F.count("*").alias("row_count")
).orderBy(
    "analysis_year"
).show(truncate=False)

print(
    f"Duplicate province-year key count: "
    f"{duplicate_province_year_keys_df.count()}"
)

duplicate_province_year_keys_df.show(truncate=False)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

province_change_window = (
    Window
    .partitionBy("geography_name")
    .orderBy("analysis_year")
)

cbsl_provincial_change_features_df = (
    cbsl_provincial_features_df

    .withColumn(
        "previous_analysis_year",
        F.lag("analysis_year", 1).over(province_change_window)
    )
    .withColumn(
        "years_since_previous_observation",
        F.when(
            F.col("previous_analysis_year").isNotNull(),
            F.col("analysis_year") - F.col("previous_analysis_year")
        )
    )

    # Previous income and expenditure values
    .withColumn(
        "previous_mean_income_per_household_monthly_lkr",
        F.lag("mean_income_per_household_monthly_lkr", 1)
        .over(province_change_window)
    )
    .withColumn(
        "previous_mean_income_per_person_monthly_lkr",
        F.lag("mean_income_per_person_monthly_lkr", 1)
        .over(province_change_window)
    )
    .withColumn(
        "previous_median_income_per_household_monthly_lkr",
        F.lag("median_income_per_household_monthly_lkr", 1)
        .over(province_change_window)
    )
    .withColumn(
        "previous_household_expenditure_monthly_lkr",
        F.lag("household_expenditure_monthly_lkr", 1)
        .over(province_change_window)
    )

    # Nominal percentage changes from 2016 to 2019
    .withColumn(
        "mean_household_income_change_percent",
        F.when(
            F.col("previous_mean_income_per_household_monthly_lkr") > 0,
            F.round(
                (
                    F.col("mean_income_per_household_monthly_lkr")
                    / F.col("previous_mean_income_per_household_monthly_lkr")
                    - 1
                ) * 100,
                4
            )
        )
    )
    .withColumn(
        "mean_per_person_income_change_percent",
        F.when(
            F.col("previous_mean_income_per_person_monthly_lkr") > 0,
            F.round(
                (
                    F.col("mean_income_per_person_monthly_lkr")
                    / F.col("previous_mean_income_per_person_monthly_lkr")
                    - 1
                ) * 100,
                4
            )
        )
    )
    .withColumn(
        "median_household_income_change_percent",
        F.when(
            F.col("previous_median_income_per_household_monthly_lkr") > 0,
            F.round(
                (
                    F.col("median_income_per_household_monthly_lkr")
                    / F.col("previous_median_income_per_household_monthly_lkr")
                    - 1
                ) * 100,
                4
            )
        )
    )
    .withColumn(
        "household_expenditure_change_percent",
        F.when(
            F.col("previous_household_expenditure_monthly_lkr") > 0,
            F.round(
                (
                    F.col("household_expenditure_monthly_lkr")
                    / F.col("previous_household_expenditure_monthly_lkr")
                    - 1
                ) * 100,
                4
            )
        )
    )

    # Point changes for ratios and shares
    .withColumn(
        "communication_expenditure_share_change_pp",
        F.round(
            F.col("expenditure_share_communication_percent")
            - F.lag("expenditure_share_communication_percent", 1)
              .over(province_change_window),
            4
        )
    )
    .withColumn(
        "education_expenditure_share_change_pp",
        F.round(
            F.col("expenditure_share_education_percent")
            - F.lag("expenditure_share_education_percent", 1)
              .over(province_change_window),
            4
        )
    )
    .withColumn(
        "transport_expenditure_share_change_pp",
        F.round(
            F.col("expenditure_share_transport_percent")
            - F.lag("expenditure_share_transport_percent", 1)
              .over(province_change_window),
            4
        )
    )
    .withColumn(
        "housing_expenditure_share_change_pp",
        F.round(
            F.col("expenditure_share_housing_percent")
            - F.lag("expenditure_share_housing_percent", 1)
              .over(province_change_window),
            4
        )
    )
    .withColumn(
        "household_gini_change_points",
        F.round(
            F.col("gini_coefficient_households")
            - F.lag("gini_coefficient_households", 1)
              .over(province_change_window),
            4
        )
    )

    .orderBy("geography_name", "analysis_year")
)

print("Province socioeconomic change features:")

cbsl_provincial_change_features_df.select(
    "geography_name",
    "analysis_year",
    "previous_analysis_year",
    "years_since_previous_observation",
    "mean_household_income_change_percent",
    "mean_per_person_income_change_percent",
    "median_household_income_change_percent",
    "household_expenditure_change_percent",
    "communication_expenditure_share_change_pp",
    "education_expenditure_share_change_pp",
    "transport_expenditure_share_change_pp",
    "housing_expenditure_share_change_pp",
    "household_gini_change_points"
).show(30, truncate=False)

In [0]:
from pyspark.sql import functions as F

print("Province socioeconomic change-feature validation:")

cbsl_provincial_change_features_df.agg(
    F.count("*").alias("row_count"),

    F.sum(
        F.col("mean_household_income_change_percent")
        .isNull().cast("int")
    ).alias("mean_income_change_null_count"),

    F.sum(
        F.col("mean_per_person_income_change_percent")
        .isNull().cast("int")
    ).alias("per_person_income_change_null_count"),

    F.sum(
        F.col("median_household_income_change_percent")
        .isNull().cast("int")
    ).alias("median_income_change_null_count"),

    F.sum(
        F.col("household_expenditure_change_percent")
        .isNull().cast("int")
    ).alias("expenditure_change_null_count"),

    F.sum(
        F.when(
            F.col("years_since_previous_observation") <= 0,
            1
        ).otherwise(0)
    ).alias("invalid_observation_gap_count"),

    F.sum(
        F.col("mean_household_income_change_percent")
        .isNotNull().cast("int")
    ).alias("valid_change_row_count"),

    F.min(
        "mean_household_income_change_percent"
    ).alias("minimum_mean_income_change_percent"),

    F.max(
        "mean_household_income_change_percent"
    ).alias("maximum_mean_income_change_percent"),

    F.min(
        "household_expenditure_change_percent"
    ).alias("minimum_expenditure_change_percent"),

    F.max(
        "household_expenditure_change_percent"
    ).alias("maximum_expenditure_change_percent")
).show(truncate=False)

print("Valid change coverage by year:")

cbsl_provincial_change_features_df.groupBy(
    "analysis_year"
).agg(
    F.countDistinct("geography_name").alias("province_count"),

    F.sum(
        F.col("mean_household_income_change_percent")
        .isNotNull().cast("int")
    ).alias("valid_change_count")
).orderBy(
    "analysis_year"
).show(truncate=False)

In [0]:
cbsl_provincial_analytical_df = (
    cbsl_provincial_change_features_df
    .select(
        # Geography and time
        "analysis_year",
        "geography_type",
        "geography_name",
        "previous_analysis_year",
        "years_since_previous_observation",

        # Household structure
        "individuals_per_household",
        "income_receivers_per_household",

        # Income levels — nominal LKR
        "mean_income_per_household_monthly_lkr",
        "mean_income_per_person_monthly_lkr",
        "median_income_per_household_monthly_lkr",

        # Inequality
        "gini_coefficient_households",

        # Expenditure level and shares
        "household_expenditure_monthly_lkr",
        "expenditure_share_communication_percent",
        "expenditure_share_education_percent",
        "expenditure_share_transport_percent",
        "expenditure_share_housing_percent",

        # Educational attainment
        "education_passed_gce_ol_percent",
        "education_passed_gce_al_and_above_percent",

        # Nominal income and expenditure changes
        "mean_household_income_change_percent",
        "mean_per_person_income_change_percent",
        "median_household_income_change_percent",
        "household_expenditure_change_percent",

        # Expenditure-share and inequality changes
        "communication_expenditure_share_change_pp",
        "education_expenditure_share_change_pp",
        "transport_expenditure_share_change_pp",
        "housing_expenditure_share_change_pp",
        "household_gini_change_points"
    )
    .orderBy(
        "geography_name",
        "analysis_year"
    )
)

print(
    f"Curated provincial socioeconomic rows: "
    f"{cbsl_provincial_analytical_df.count()}"
)

print(
    f"Curated provincial socioeconomic columns: "
    f"{len(cbsl_provincial_analytical_df.columns)}"
)

print("\nColumns:")
print(cbsl_provincial_analytical_df.columns)

print("\nSchema:")
cbsl_provincial_analytical_df.printSchema()

In [0]:
from pyspark.sql import functions as F

duplicate_provincial_analytical_keys_df = (
    cbsl_provincial_analytical_df
    .groupBy(
        "analysis_year",
        "geography_type",
        "geography_name"
    )
    .count()
    .filter(F.col("count") > 1)
)

print("Curated provincial socioeconomic final validation:")

cbsl_provincial_analytical_df.agg(
    F.count("*").alias("row_count"),

    F.countDistinct("geography_name")
    .alias("province_count"),

    F.countDistinct("analysis_year")
    .alias("year_count"),

    F.sum(
        F.col("mean_income_per_household_monthly_lkr")
        .isNull().cast("int")
    ).alias("mean_household_income_null_count"),

    F.sum(
        F.col("household_expenditure_monthly_lkr")
        .isNull().cast("int")
    ).alias("household_expenditure_null_count"),

    F.sum(
        F.col("expenditure_share_communication_percent")
        .isNull().cast("int")
    ).alias("communication_share_null_count"),

    F.sum(
        F.col("gini_coefficient_households")
        .isNull().cast("int")
    ).alias("household_gini_null_count"),

    F.sum(
        F.col("mean_household_income_change_percent")
        .isNotNull().cast("int")
    ).alias("valid_income_change_count"),

    F.sum(
        F.col("household_expenditure_change_percent")
        .isNotNull().cast("int")
    ).alias("valid_expenditure_change_count")
).show(truncate=False)

print(
    f"Duplicate province-year key count: "
    f"{duplicate_provincial_analytical_keys_df.count()}"
)

duplicate_provincial_analytical_keys_df.show(truncate=False)

print("Records and change availability per province:")

cbsl_provincial_analytical_df.groupBy(
    "geography_name"
).agg(
    F.count("*").alias("row_count"),

    F.countDistinct("analysis_year")
    .alias("year_count"),

    F.sum(
        F.col("mean_household_income_change_percent")
        .isNotNull().cast("int")
    ).alias("valid_income_change_count"),

    F.sum(
        F.col("household_expenditure_change_percent")
        .isNotNull().cast("int")
    ).alias("valid_expenditure_change_count")
).orderBy(
    "geography_name"
).show(20, truncate=False)

In [0]:
from pyspark.sql import functions as F

telecom_availability_df = (
    cbsl_telecom_analytical_df
    .agg(
        F.lit("CBSL annual telecom").alias("dataset_name"),
        F.lit("national").alias("geography_level"),
        F.count("*").alias("observation_count"),
        F.countDistinct("year").alias("distinct_time_count"),
        F.min("year").alias("minimum_year"),
        F.max("year").alias("maximum_year"),
        F.sum(
            F.col("yoy_growth_features_available").cast("int")
        ).alias("usable_change_observation_count"),
        F.lit(
            "Annual data; total internet connections include mobile"
        ).alias("analysis_note")
    )
)

hies_availability_df = (
    cbsl_hies_analytical_df
    .agg(
        F.lit("CBSL national HIES").alias("dataset_name"),
        F.lit("national").alias("geography_level"),
        F.count("*").alias("observation_count"),
        F.countDistinct("representative_year").alias("distinct_time_count"),
        F.min("representative_year").alias("minimum_year"),
        F.max("representative_year").alias("maximum_year"),
        F.sum(
            F.col("overlaps_cbsl_telecom_period").cast("int")
        ).alias("usable_change_observation_count"),
        F.lit(
            "Irregular survey periods; only 2013, 2016 and 2019 overlap telecom"
        ).alias("analysis_note")
    )
)

national_poverty_availability_df = (
    cbsl_poverty_national_analytical_df
    .agg(
        F.lit("CBSL national poverty").alias("dataset_name"),
        F.lit("national").alias("geography_level"),
        F.count("*").alias("observation_count"),
        F.countDistinct("representative_year").alias("distinct_time_count"),
        F.min("representative_year").alias("minimum_year"),
        F.max("representative_year").alias("maximum_year"),
        F.sum(
            F.col("poverty_headcount_change_pp").isNotNull().cast("int")
        ).alias("usable_change_observation_count"),
        F.lit(
            "2019 poverty definitions remain separate; only 2 safe historical changes"
        ).alias("analysis_note")
    )
)

province_poverty_availability_df = (
    cbsl_poverty_province_analytical_df
    .agg(
        F.lit("CBSL province poverty").alias("dataset_name"),
        F.lit("province").alias("geography_level"),
        F.count("*").alias("observation_count"),
        F.countDistinct("representative_year").alias("distinct_time_count"),
        F.min("representative_year").alias("minimum_year"),
        F.max("representative_year").alias("maximum_year"),
        F.sum(
            F.col("poverty_headcount_change_pp").isNotNull().cast("int")
        ).alias("usable_change_observation_count"),
        F.lit(
            "18 safe province-change observations; 2019(b) excluded from changes"
        ).alias("analysis_note")
    )
)

provincial_socioeconomic_availability_df = (
    cbsl_provincial_analytical_df
    .agg(
        F.lit("CBSL provincial socioeconomic").alias("dataset_name"),
        F.lit("province").alias("geography_level"),
        F.count("*").alias("observation_count"),
        F.countDistinct("analysis_year").alias("distinct_time_count"),
        F.min("analysis_year").alias("minimum_year"),
        F.max("analysis_year").alias("maximum_year"),
        F.sum(
            F.col("mean_household_income_change_percent")
            .isNotNull()
            .cast("int")
        ).alias("usable_change_observation_count"),
        F.lit(
            "Only 2016 and 2019; 9 province-level change observations"
        ).alias("analysis_note")
    )
)

cbsl_observation_availability_df = (
    telecom_availability_df
    .unionByName(hies_availability_df)
    .unionByName(national_poverty_availability_df)
    .unionByName(province_poverty_availability_df)
    .unionByName(provincial_socioeconomic_availability_df)
    .orderBy("geography_level", "dataset_name")
)

print("CBSL analytical observation-availability matrix:")

cbsl_observation_availability_df.show(
    20,
    truncate=False
)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

telecom_year_window = Window.orderBy("year")
three_year_window = telecom_year_window.rowsBetween(-2, 0)

telecom_silver_df = spark.table(
    "workspace.census360.silver_cbsl_telecommunication_services"
)

telecom_wide_df = (
    telecom_silver_df
    .groupBy("year")
    .pivot("metric_code")
    .agg(F.first("value_numeric"))
)

telecom_status_df = (
    telecom_silver_df
    .groupBy("year")
    .agg(
        F.first("data_status").alias("data_status"),
        F.countDistinct("data_status").alias("status_count")
    )
)

telecom_base_df = (
    telecom_wide_df
    .join(telecom_status_df, on="year", how="left")
)

telecom_features_df = (
    telecom_base_df

    .withColumn(
        "internet_connections_previous_year",
        F.lag("internet_connections_total", 1).over(telecom_year_window)
    )
    .withColumn(
        "fixed_subscriber_base_previous_year_thousand",
        F.lag("fixed_subscriber_base_thousand", 1).over(telecom_year_window)
    )

    .withColumn(
        "internet_connections_yoy_change",
        F.col("internet_connections_total")
        - F.col("internet_connections_previous_year")
    )
    .withColumn(
        "fixed_subscriber_base_yoy_change_thousand",
        F.col("fixed_subscriber_base_thousand")
        - F.col("fixed_subscriber_base_previous_year_thousand")
    )

    .withColumn(
        "internet_connections_yoy_growth_percent",
        F.when(
            F.col("internet_connections_previous_year") > 0,
            F.round(
                (
                    F.col("internet_connections_yoy_change")
                    / F.col("internet_connections_previous_year")
                ) * 100,
                4
            )
        )
    )
    .withColumn(
        "fixed_subscriber_base_yoy_growth_calculated_percent",
        F.when(
            F.col("fixed_subscriber_base_previous_year_thousand") > 0,
            F.round(
                (
                    F.col("fixed_subscriber_base_yoy_change_thousand")
                    / F.col("fixed_subscriber_base_previous_year_thousand")
                ) * 100,
                4
            )
        )
    )

    .withColumn(
        "telephone_penetration_previous_year",
        F.lag("telephone_penetration_per_100", 1).over(telecom_year_window)
    )
    .withColumn(
        "cellular_penetration_previous_year",
        F.lag("cellular_penetration_per_100", 1).over(telecom_year_window)
    )
    .withColumn(
        "telephone_penetration_change_pp",
        F.round(
            F.col("telephone_penetration_per_100")
            - F.col("telephone_penetration_previous_year"),
            4
        )
    )
    .withColumn(
        "cellular_penetration_change_pp",
        F.round(
            F.col("cellular_penetration_per_100")
            - F.col("cellular_penetration_previous_year"),
            4
        )
    )

    .withColumn(
        "fixed_component_total_thousand",
        F.col("wireline_telephones_thousand")
        + F.col("wireless_local_loop_thousand")
    )
    .withColumn(
        "wireline_share_of_fixed_components_percent",
        F.when(
            F.col("fixed_component_total_thousand") > 0,
            F.round(
                (
                    F.col("wireline_telephones_thousand")
                    / F.col("fixed_component_total_thousand")
                ) * 100,
                4
            )
        )
    )
    .withColumn(
        "wireless_local_loop_share_of_fixed_components_percent",
        F.when(
            F.col("fixed_component_total_thousand") > 0,
            F.round(
                (
                    F.col("wireless_local_loop_thousand")
                    / F.col("fixed_component_total_thousand")
                ) * 100,
                4
            )
        )
    )
    .withColumn(
        "fixed_component_difference_thousand",
        F.col("fixed_subscriber_base_thousand")
        - F.col("fixed_component_total_thousand")
    )
    .withColumn(
        "fixed_component_match_flag",
        F.abs(F.col("fixed_component_difference_thousand")) <= 0.01
    )

    .withColumn(
        "log_internet_connections_total",
        F.when(
            F.col("internet_connections_total") > 0,
            F.round(F.log("internet_connections_total"), 6)
        )
    )
    .withColumn(
        "log_fixed_subscriber_base_thousand",
        F.when(
            F.col("fixed_subscriber_base_thousand") > 0,
            F.round(F.log("fixed_subscriber_base_thousand"), 6)
        )
    )

    .withColumn(
        "internet_connections_lag_1_year",
        F.lag("internet_connections_total", 1).over(telecom_year_window)
    )
    .withColumn(
        "internet_connections_lag_2_years",
        F.lag("internet_connections_total", 2).over(telecom_year_window)
    )
    .withColumn(
        "internet_growth_percent_lag_1_year",
        F.lag("internet_connections_yoy_growth_percent", 1)
        .over(telecom_year_window)
    )
    .withColumn(
        "internet_growth_percent_lag_2_years",
        F.lag("internet_connections_yoy_growth_percent", 2)
        .over(telecom_year_window)
    )
    .withColumn(
        "fixed_subscriber_base_lag_1_year_thousand",
        F.lag("fixed_subscriber_base_thousand", 1)
        .over(telecom_year_window)
    )
    .withColumn(
        "fixed_subscriber_base_lag_2_years_thousand",
        F.lag("fixed_subscriber_base_thousand", 2)
        .over(telecom_year_window)
    )
    .withColumn(
        "fixed_growth_percent_lag_1_year",
        F.lag(
            "fixed_subscriber_base_yoy_growth_calculated_percent",
            1
        ).over(telecom_year_window)
    )
    .withColumn(
        "fixed_growth_percent_lag_2_years",
        F.lag(
            "fixed_subscriber_base_yoy_growth_calculated_percent",
            2
        ).over(telecom_year_window)
    )

    .withColumn(
        "internet_growth_3yr_observation_count",
        F.count("internet_connections_yoy_growth_percent")
        .over(three_year_window)
    )
    .withColumn(
        "internet_growth_3yr_trailing_average_percent",
        F.when(
            F.col("internet_growth_3yr_observation_count") == 3,
            F.round(
                F.avg("internet_connections_yoy_growth_percent")
                .over(three_year_window),
                4
            )
        )
    )
    .withColumn(
        "fixed_growth_3yr_observation_count",
        F.count(
            "fixed_subscriber_base_yoy_growth_calculated_percent"
        ).over(three_year_window)
    )
    .withColumn(
        "fixed_growth_3yr_trailing_average_percent",
        F.when(
            F.col("fixed_growth_3yr_observation_count") == 3,
            F.round(
                F.avg(
                    "fixed_subscriber_base_yoy_growth_calculated_percent"
                ).over(three_year_window),
                4
            )
        )
    )

    .withColumn(
        "is_published_observation",
        F.col("data_status") == "published"
    )
    .withColumn(
        "is_revised_observation",
        F.col("data_status") == "revised"
    )
    .withColumn(
        "is_provisional_observation",
        F.col("data_status") == "provisional"
    )
    .withColumn(
        "yoy_growth_features_available",
        F.col("internet_connections_yoy_growth_percent").isNotNull()
        & F.col(
            "fixed_subscriber_base_yoy_growth_calculated_percent"
        ).isNotNull()
    )
    .withColumn(
        "one_year_growth_lags_available",
        F.col("internet_growth_percent_lag_1_year").isNotNull()
        & F.col("fixed_growth_percent_lag_1_year").isNotNull()
    )
    .withColumn(
        "two_year_growth_lags_available",
        F.col("internet_growth_percent_lag_2_years").isNotNull()
        & F.col("fixed_growth_percent_lag_2_years").isNotNull()
    )
    .withColumn(
        "three_year_rolling_features_available",
        F.col(
            "internet_growth_3yr_trailing_average_percent"
        ).isNotNull()
        & F.col(
            "fixed_growth_3yr_trailing_average_percent"
        ).isNotNull()
    )
    .withColumn(
        "internet_metric_scope",
        F.lit("total_internet_connections_including_mobile")
    )
    .withColumn(
        "fixed_metric_scope",
        F.lit("fixed_access_subscriber_base_not_fixed_broadband")
    )
)

cbsl_telecom_analytical_df = (
    telecom_features_df
    .select(
        "year",
        "data_status",
        "is_published_observation",
        "is_revised_observation",
        "is_provisional_observation",
        "internet_connections_total",
        "fixed_subscriber_base_thousand",
        "wireline_telephones_thousand",
        "wireless_local_loop_thousand",
        "telephone_penetration_per_100",
        "cellular_phones_thousand",
        "cellular_penetration_per_100",
        "public_pay_phone_booths",
        "internet_connections_yoy_change",
        "internet_connections_yoy_growth_percent",
        "fixed_subscriber_base_yoy_change_thousand",
        "fixed_subscriber_growth_percent",
        "fixed_subscriber_base_yoy_growth_calculated_percent",
        "telephone_penetration_change_pp",
        "cellular_penetration_change_pp",
        "wireline_share_of_fixed_components_percent",
        "wireless_local_loop_share_of_fixed_components_percent",
        "fixed_component_difference_thousand",
        "fixed_component_match_flag",
        "log_internet_connections_total",
        "log_fixed_subscriber_base_thousand",
        "internet_connections_lag_1_year",
        "internet_connections_lag_2_years",
        "internet_growth_percent_lag_1_year",
        "internet_growth_percent_lag_2_years",
        "fixed_subscriber_base_lag_1_year_thousand",
        "fixed_subscriber_base_lag_2_years_thousand",
        "fixed_growth_percent_lag_1_year",
        "fixed_growth_percent_lag_2_years",
        "internet_growth_3yr_trailing_average_percent",
        "fixed_growth_3yr_trailing_average_percent",
        "yoy_growth_features_available",
        "one_year_growth_lags_available",
        "two_year_growth_lags_available",
        "three_year_rolling_features_available",
        "internet_metric_scope",
        "fixed_metric_scope"
    )
    .orderBy("year")
)

print(
    f"Telecom analytical DataFrame recreated: "
    f"{cbsl_telecom_analytical_df.count()} rows, "
    f"{len(cbsl_telecom_analytical_df.columns)} columns"
)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

hies_silver_df = spark.table(
    "workspace.census360.silver_cbsl_household_income_hies"
)

hies_metadata_df = (
    hies_silver_df
    .select(
        "survey_period",
        "survey_period_type",
        "period_start_year",
        "period_end_year"
    )
    .distinct()
)

hies_wide_df = (
    hies_silver_df
    .groupBy("survey_period")
    .pivot("metric_code")
    .agg(F.first("value_numeric"))
)

hies_window = Window.orderBy("representative_year")

hies_base_df = (
    hies_metadata_df
    .join(
        hies_wide_df,
        on="survey_period",
        how="inner"
    )
    .withColumn(
        "representative_year",
        F.col("period_end_year")
    )
    .withColumn(
        "alignment_method",
        F.lit("survey_period_end_year")
    )
    .withColumn(
        "is_original_single_year_observation",
        F.col("survey_period_type") == "single_year"
    )
    .withColumn(
        "is_multi_year_survey_observation",
        F.col("survey_period_type") == "multi_year"
    )
    .withColumn(
        "is_derived_annual_value",
        F.lit(False)
    )
    .withColumn(
        "overlaps_cbsl_telecom_period",
        F.col("representative_year").between(2013, 2024)
    )
    .withColumn(
        "previous_survey_period",
        F.lag("survey_period", 1).over(hies_window)
    )
    .withColumn(
        "previous_representative_year",
        F.lag("representative_year", 1).over(hies_window)
    )
    .withColumn(
        "years_since_previous_survey",
        F.when(
            F.col("previous_representative_year").isNotNull(),
            F.col("representative_year")
            - F.col("previous_representative_year")
        )
    )
)

hies_change_df = (
    hies_base_df

    .withColumn(
        "previous_mean_household_income_monthly_lkr",
        F.lag("mean_household_income_monthly_lkr", 1)
        .over(hies_window)
    )
    .withColumn(
        "previous_median_household_income_monthly_lkr",
        F.lag("median_household_income_monthly_lkr", 1)
        .over(hies_window)
    )
    .withColumn(
        "previous_mean_per_capita_income_monthly_lkr",
        F.lag("mean_per_capita_income_monthly_lkr", 1)
        .over(hies_window)
    )
    .withColumn(
        "previous_mean_household_expenditure_monthly_lkr",
        F.lag("mean_household_expenditure_monthly_lkr", 1)
        .over(hies_window)
    )
    .withColumn(
        "previous_food_ratio_percent",
        F.lag("food_ratio_percent", 1).over(hies_window)
    )
    .withColumn(
        "previous_gini_household_income",
        F.lag("gini_household_income", 1).over(hies_window)
    )
    .withColumn(
        "previous_poverty_headcount_percent",
        F.lag("poverty_headcount_ratio_2002_based_percent", 1)
        .over(hies_window)
    )

    .withColumn(
        "mean_household_income_change_percent",
        F.when(
            F.col("previous_mean_household_income_monthly_lkr") > 0,
            F.round(
                (
                    F.col("mean_household_income_monthly_lkr")
                    / F.col("previous_mean_household_income_monthly_lkr")
                    - 1
                ) * 100,
                4
            )
        )
    )
    .withColumn(
        "median_household_income_change_percent",
        F.when(
            F.col("previous_median_household_income_monthly_lkr") > 0,
            F.round(
                (
                    F.col("median_household_income_monthly_lkr")
                    / F.col("previous_median_household_income_monthly_lkr")
                    - 1
                ) * 100,
                4
            )
        )
    )
    .withColumn(
        "mean_per_capita_income_change_percent",
        F.when(
            F.col("previous_mean_per_capita_income_monthly_lkr") > 0,
            F.round(
                (
                    F.col("mean_per_capita_income_monthly_lkr")
                    / F.col("previous_mean_per_capita_income_monthly_lkr")
                    - 1
                ) * 100,
                4
            )
        )
    )
    .withColumn(
        "mean_household_expenditure_change_percent",
        F.when(
            F.col("previous_mean_household_expenditure_monthly_lkr") > 0,
            F.round(
                (
                    F.col("mean_household_expenditure_monthly_lkr")
                    / F.col("previous_mean_household_expenditure_monthly_lkr")
                    - 1
                ) * 100,
                4
            )
        )
    )

    .withColumn(
        "mean_household_income_annualized_growth_percent",
        F.when(
            (F.col("previous_mean_household_income_monthly_lkr") > 0)
            & (F.col("years_since_previous_survey") > 0),
            F.round(
                (
                    F.exp(
                        F.log(
                            F.col("mean_household_income_monthly_lkr")
                            / F.col("previous_mean_household_income_monthly_lkr")
                        )
                        / F.col("years_since_previous_survey")
                    ) - 1
                ) * 100,
                4
            )
        )
    )
    .withColumn(
        "mean_household_expenditure_annualized_growth_percent",
        F.when(
            (F.col("previous_mean_household_expenditure_monthly_lkr") > 0)
            & (F.col("years_since_previous_survey") > 0),
            F.round(
                (
                    F.exp(
                        F.log(
                            F.col("mean_household_expenditure_monthly_lkr")
                            / F.col("previous_mean_household_expenditure_monthly_lkr")
                        )
                        / F.col("years_since_previous_survey")
                    ) - 1
                ) * 100,
                4
            )
        )
    )

    .withColumn(
        "food_ratio_change_pp",
        F.round(
            F.col("food_ratio_percent")
            - F.col("previous_food_ratio_percent"),
            4
        )
    )
    .withColumn(
        "income_gini_change_points",
        F.round(
            F.col("gini_household_income")
            - F.col("previous_gini_household_income"),
            4
        )
    )
    .withColumn(
        "poverty_headcount_change_pp",
        F.round(
            F.col("poverty_headcount_ratio_2002_based_percent")
            - F.col("previous_poverty_headcount_percent"),
            4
        )
    )
)

cbsl_hies_analytical_df = (
    hies_change_df
    .select(
        "survey_period",
        "survey_period_type",
        "period_start_year",
        "period_end_year",
        "representative_year",
        "previous_survey_period",
        "previous_representative_year",
        "years_since_previous_survey",
        "alignment_method",
        "is_original_single_year_observation",
        "is_multi_year_survey_observation",
        "is_derived_annual_value",
        "overlaps_cbsl_telecom_period",
        "mean_household_income_monthly_lkr",
        "median_household_income_monthly_lkr",
        "mean_per_capita_income_monthly_lkr",
        "mean_household_expenditure_monthly_lkr",
        "food_drink_expenditure_monthly_lkr",
        "non_food_expenditure_monthly_lkr",
        "food_ratio_percent",
        "gini_household_income",
        "poverty_headcount_ratio_2002_based_percent",
        "household_size",
        "income_receivers_per_household",
        "mean_household_income_change_percent",
        "median_household_income_change_percent",
        "mean_per_capita_income_change_percent",
        "mean_household_expenditure_change_percent",
        "mean_household_income_annualized_growth_percent",
        "mean_household_expenditure_annualized_growth_percent",
        "food_ratio_change_pp",
        "income_gini_change_points",
        "poverty_headcount_change_pp"
    )
    .orderBy("representative_year")
)

print(
    f"HIES analytical DataFrame recreated: "
    f"{cbsl_hies_analytical_df.count()} rows, "
    f"{len(cbsl_hies_analytical_df.columns)} columns"
)

In [0]:
hies_target_table = "workspace.census360.gold_cbsl_hies_analytical_v1"

if spark.catalog.tableExists(hies_target_table):
    print(f"Table already exists. Nothing was overwritten: {hies_target_table}")
else:
    (
        cbsl_hies_analytical_df
        .write
        .format("delta")
        .mode("errorifexists")
        .saveAsTable(hies_target_table)
    )

    saved_hies_df = spark.table(hies_target_table)

    print(f"Table created: {hies_target_table}")
    print(f"Rows: {saved_hies_df.count()}")
    print(f"Columns: {len(saved_hies_df.columns)}")

In [0]:
hies_target_table = "workspace.census360.gold_cbsl_hies_analytical_v1"

if spark.catalog.tableExists(hies_target_table):
    print(f"Table already exists. Nothing was written: {hies_target_table}")
else:
    (
        cbsl_hies_analytical_df
        .write
        .format("delta")
        .mode("append")
        .saveAsTable(hies_target_table)
    )

    saved_hies_df = spark.table(hies_target_table)

    print(f"Table created: {hies_target_table}")
    print(f"Rows: {saved_hies_df.count()}")
    print(f"Columns: {len(saved_hies_df.columns)}")

In [0]:
telecom_target_table = "workspace.census360.gold_cbsl_telecom_analytical_v1"

if spark.catalog.tableExists(telecom_target_table):
    print(f"Table already exists. Nothing was written: {telecom_target_table}")
else:
    (
        cbsl_telecom_analytical_df
        .write
        .format("delta")
        .mode("append")
        .saveAsTable(telecom_target_table)
    )

    saved_telecom_df = spark.table(telecom_target_table)

    print(f"Table created: {telecom_target_table}")
    print(f"Rows: {saved_telecom_df.count()}")
    print(f"Columns: {len(saved_telecom_df.columns)}")

In [0]:
national_poverty_target_table = (
    "workspace.census360.gold_cbsl_poverty_national_analytical_v1"
)

if spark.catalog.tableExists(national_poverty_target_table):
    print(
        f"Table already exists. Nothing was written: "
        f"{national_poverty_target_table}"
    )
else:
    (
        cbsl_poverty_national_analytical_df
        .write
        .format("delta")
        .mode("append")
        .saveAsTable(national_poverty_target_table)
    )

    saved_national_poverty_df = spark.table(
        national_poverty_target_table
    )

    print(f"Table created: {national_poverty_target_table}")
    print(f"Rows: {saved_national_poverty_df.count()}")
    print(f"Columns: {len(saved_national_poverty_df.columns)}")

In [0]:
province_poverty_target_table = (
    "workspace.census360.gold_cbsl_poverty_province_analytical_v1"
)

if spark.catalog.tableExists(province_poverty_target_table):
    print(
        f"Table already exists. Nothing was written: "
        f"{province_poverty_target_table}"
    )
else:
    (
        cbsl_poverty_province_analytical_df
        .write
        .format("delta")
        .mode("append")
        .saveAsTable(province_poverty_target_table)
    )

    saved_province_poverty_df = spark.table(
        province_poverty_target_table
    )

    print(f"Table created: {province_poverty_target_table}")
    print(f"Rows: {saved_province_poverty_df.count()}")
    print(f"Columns: {len(saved_province_poverty_df.columns)}")

In [0]:
provincial_socioeconomic_target_table = (
    "workspace.census360.gold_cbsl_provincial_socioeconomic_analytical_v1"
)

if spark.catalog.tableExists(provincial_socioeconomic_target_table):
    print(
        f"Table already exists. Nothing was written: "
        f"{provincial_socioeconomic_target_table}"
    )
else:
    (
        cbsl_provincial_analytical_df
        .write
        .format("delta")
        .mode("append")
        .saveAsTable(provincial_socioeconomic_target_table)
    )

    saved_provincial_socioeconomic_df = spark.table(
        provincial_socioeconomic_target_table
    )

    print(f"Table created: {provincial_socioeconomic_target_table}")
    print(f"Rows: {saved_provincial_socioeconomic_df.count()}")
    print(f"Columns: {len(saved_provincial_socioeconomic_df.columns)}")

In [0]:
cbsl_gold_tables = [
    "workspace.census360.gold_cbsl_telecom_analytical_v1",
    "workspace.census360.gold_cbsl_hies_analytical_v1",
    "workspace.census360.gold_cbsl_poverty_national_analytical_v1",
    "workspace.census360.gold_cbsl_poverty_province_analytical_v1",
    "workspace.census360.gold_cbsl_provincial_socioeconomic_analytical_v1"
]

print("CBSL Gold analytical table verification:\n")

for table_name in cbsl_gold_tables:
    if spark.catalog.tableExists(table_name):
        table_df = spark.table(table_name)

        print(f"Table: {table_name}")
        print("Exists: True")
        print(f"Rows: {table_df.count()}")
        print(f"Columns: {len(table_df.columns)}")
        print("-" * 70)
    else:
        print(f"Table: {table_name}")
        print("Exists: False")
        print("-" * 70)